# CDPL Baseline Analysis PAMAP2 Notebook


## Colab setup
Mount Google Drive when running in Colab.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## Imports
Load the Python packages used throughout the notebook.

In [ ]:
drive.mount('/content/drive')

from __future__ import annotations

import copy
import math
import os
import random
from collections import defaultdict
from dataclasses import dataclass, asdict
from pathlib import Path
from typing import Dict, List, Tuple, Optional

import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score, f1_score
from sklearn.preprocessing import StandardScaler
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader


## Reproducibility
Seed configuration and reproducibility utilities.

In [ ]:
# ============================================================
# Reproducibility
# ============================================================

def set_seed(seed: int = 42) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    torch.backends.cudnn.benchmark = True
    torch.backends.cudnn.deterministic = False
    if hasattr(torch, "set_float32_matmul_precision"):
        torch.set_float32_matmul_precision("high")


## Configuration
Experiment hyperparameters and runtime configuration collected in the `Config` dataclass.

In [ ]:
# ============================================================
# Configuration
# ============================================================

@dataclass
class Config:
    # ----- data -----
    data_dir: str = "/content/PAMAP2_Dataset/Protocol"
    subject_glob: str = "subject*.dat"
    valid_activity_ids: Tuple[int, ...] = (1, 2, 3, 4, 5, 6, 7, 12, 13, 16, 17, 24)

    seq_len: int = 100
    stride: int = 50
    drop_mixed_windows: bool = True
    purge_gap_raw: int = 50  # active gap between sub-splits inside the same constant-label segment

    train_subject_train_ratio: float = 0.85
    train_subject_val_ratio: float = 0.15

    heldout_support_ratio: float = 0.20
    heldout_adapt_val_ratio: float = 0.20
    heldout_test_ratio: float = 0.60

    min_windows_per_partition: int = 8

    # ----- optimization -----
    batch_size: int = 128
    personal_batch_size: int = 64
    num_workers: int = 0
    pin_memory: bool = True

    rounds: int = 15
    local_epochs: int = 6
    personalization_epochs: int = 10

    lr_encoder: float = 3e-4
    lr_personal: float = 1e-2
    weight_decay: float = 1e-4
    grad_clip: float = 1.0

    lambda_align: float = 0.7
    lambda_A: float = 1e-3
    lambda_B: float = 5e-3

    server_geom_steps: int = 20
    server_geom_lr: float = 2e-2
    server_alt_iters: int = 3

    # ----- model -----
    d_model: int = 192
    emb_dim: int = 192
    n_heads: int = 6
    n_layers: int = 3
    ff_dim: int = 384
    dropout: float = 0.2
    proto_rank: int = 8

    # ----- misc -----
    ece_bins: int = 15
    seed: int = 42
    amp: bool = True
    device: str = "cuda" if torch.cuda.is_available() else "cpu"
    save_dir: str = "./ccd_pamap2_runs"

    # ----- stabilization / scoring -----
    class_weight_power: float = 0.5
    class_weight_max: float = 4.0
    min_proto_samples_per_class: int = 4

    tau_init: float = 8.0
    tau_min: float = 1.5
    tau_max: float = 20.0
    lambda_tau: float = 1e-4   # reduced from 5e-4

    lambda_personal_anchor: float = 3e-4   # reduced from 2e-3

    server_proto_momentum: float = 0.85
    server_basis_momentum: float = 0.80

    client_val_weight_power: float = 0.5
    round_score_f1_w: float = 0.70
    round_score_acc_w: float = 0.30
    round_score_ece_w: float = 0.05

    # ----- adaptive personalization -----
    personal_tau_lr_mult: float = 0.25
    personalization_epochs_min: int = 6
    personalization_epochs_max: int = 14
    personalization_patience: int = 3
    hard_subject_f1_threshold: float = 0.55
    easy_subject_f1_threshold: float = 0.78

    # ----- personalization gating -----
    personalization_gate_score_margin: float = 0.002
    personalization_gate_score_margin_easy: float = 0.010
    personalization_gate_f1_margin: float = 0.000

    # for faster debugging
    run_single_heldout: Optional[str] = None  # e.g. "subject101"

    def __post_init__(self) -> None:
        assert abs(self.train_subject_train_ratio + self.train_subject_val_ratio - 1.0) < 1e-8
        assert abs(
            self.heldout_support_ratio + self.heldout_adapt_val_ratio + self.heldout_test_ratio - 1.0
        ) < 1e-8
        os.makedirs(self.save_dir, exist_ok=True)


## PAMAP2 columns
Column definitions and feature selection helpers for the PAMAP2 dataset.

In [ ]:
# ============================================================
# PAMAP2 columns
# ============================================================

def pamap2_columns() -> List[str]:
    return [
        "timestamp",
        "activity_id",
        "heart_rate",
        "hand_temperature",
        "hand_acc16_x",
        "hand_acc16_y",
        "hand_acc16_z",
        "hand_acc6_x",
        "hand_acc6_y",
        "hand_acc6_z",
        "hand_gyro_x",
        "hand_gyro_y",
        "hand_gyro_z",
        "hand_mag_x",
        "hand_mag_y",
        "hand_mag_z",
        "hand_orient_1",
        "hand_orient_2",
        "hand_orient_3",
        "hand_orient_4",
        "chest_temperature",
        "chest_acc16_x",
        "chest_acc16_y",
        "chest_acc16_z",
        "chest_acc6_x",
        "chest_acc6_y",
        "chest_acc6_z",
        "chest_gyro_x",
        "chest_gyro_y",
        "chest_gyro_z",
        "chest_mag_x",
        "chest_mag_y",
        "chest_mag_z",
        "chest_orient_1",
        "chest_orient_2",
        "chest_orient_3",
        "chest_orient_4",
        "ankle_temperature",
        "ankle_acc16_x",
        "ankle_acc16_y",
        "ankle_acc16_z",
        "ankle_acc6_x",
        "ankle_acc6_y",
        "ankle_acc6_z",
        "ankle_gyro_x",
        "ankle_gyro_y",
        "ankle_gyro_z",
        "ankle_mag_x",
        "ankle_mag_y",
        "ankle_mag_z",
        "ankle_orient_1",
        "ankle_orient_2",
        "ankle_orient_3",
        "ankle_orient_4",
    ]


def feature_columns() -> List[str]:
    cols = pamap2_columns()
    return [c for c in cols if c not in ("timestamp", "activity_id")]


## Metrics
Evaluation metrics, aggregation helpers, and personalization score utilities.

In [ ]:
# ============================================================
# Metrics
# ============================================================

def expected_calibration_error(
    probs: np.ndarray,
    labels: np.ndarray,
    n_bins: int = 15,
) -> float:
    confidences = probs.max(axis=1)
    predictions = probs.argmax(axis=1)
    accuracies = (predictions == labels).astype(np.float32)

    bin_edges = np.linspace(0.0, 1.0, n_bins + 1)
    ece = 0.0
    for i in range(n_bins):
        lo, hi = bin_edges[i], bin_edges[i + 1]
        if i == n_bins - 1:
            mask = (confidences >= lo) & (confidences <= hi)
        else:
            mask = (confidences >= lo) & (confidences < hi)
        if mask.any():
            acc_bin = accuracies[mask].mean()
            conf_bin = confidences[mask].mean()
            ece += float(mask.mean()) * abs(float(acc_bin) - float(conf_bin))
    return float(ece)


def multiclass_brier_score(probs: np.ndarray, labels: np.ndarray, num_classes: int) -> float:
    one_hot = np.eye(num_classes, dtype=np.float32)[labels]
    return float(np.mean(np.sum((probs - one_hot) ** 2, axis=1)))


def summarize_probs(
    probs: np.ndarray,
    labels: np.ndarray,
    num_classes: int,
    ece_bins: int,
) -> Dict[str, float]:
    preds = probs.argmax(axis=1)
    return {
        "acc": float(accuracy_score(labels, preds)),
        "f1": float(f1_score(labels, preds, average="macro", zero_division=0)),
        "ece": expected_calibration_error(probs, labels, n_bins=ece_bins),
        "brier": multiclass_brier_score(probs, labels, num_classes=num_classes),
    }

def weighted_mean(values: List[float], weights: List[float]) -> float:
    if len(values) == 0:
        return 0.0
    w = np.asarray(weights, dtype=np.float64)
    v = np.asarray(values, dtype=np.float64)
    w = w / max(w.sum(), 1e-12)
    return float(np.sum(w * v))


def aggregate_round_metrics(
    per_client_metrics: Dict[str, Dict[str, float]],
    per_client_val_sizes: Dict[str, int],
    cfg: Config,
) -> Tuple[float, float, float, float]:
    client_ids = sorted(per_client_metrics.keys())
    weights = [max(1, per_client_val_sizes[sid]) ** cfg.client_val_weight_power for sid in client_ids]

    mean_f1 = weighted_mean([per_client_metrics[sid]["f1"] for sid in client_ids], weights)
    mean_acc = weighted_mean([per_client_metrics[sid]["acc"] for sid in client_ids], weights)
    mean_ece = weighted_mean([per_client_metrics[sid]["ece"] for sid in client_ids], weights)

    score = (
        cfg.round_score_f1_w * mean_f1
        + cfg.round_score_acc_w * mean_acc
        - cfg.round_score_ece_w * mean_ece
    )
    return mean_f1, mean_acc, mean_ece, score

def choose_personalization_hparams(
    init_f1: float,
    cfg: Config,
) -> Dict[str, float]:
    """
    Hard subjects get more epochs and weaker anchoring.
    Easy subjects get fewer epochs and slightly stronger anchoring.
    """
    if init_f1 < cfg.hard_subject_f1_threshold:
        return {
            "epochs": cfg.personalization_epochs_max,
            "anchor_w": cfg.lambda_personal_anchor * 0.35,
            "lr_A": cfg.lr_personal,
            "lr_tau": cfg.lr_personal * cfg.personal_tau_lr_mult * 0.75,
            "warm_epochs": 2,
        }
    elif init_f1 > cfg.easy_subject_f1_threshold:
        return {
            "epochs": cfg.personalization_epochs_min,
            "anchor_w": cfg.lambda_personal_anchor * 2.0,
            "lr_A": cfg.lr_personal * 0.85,
            "lr_tau": cfg.lr_personal * cfg.personal_tau_lr_mult * 0.50,
            "warm_epochs": 1,
        }
    else:
        return {
            "epochs": int(round(0.5 * (cfg.personalization_epochs_min + cfg.personalization_epochs_max))),
            "anchor_w": cfg.lambda_personal_anchor,
            "lr_A": cfg.lr_personal,
            "lr_tau": cfg.lr_personal * cfg.personal_tau_lr_mult,
            "warm_epochs": 2,
        }

def personalization_selection_score(metrics: Dict[str, float]) -> float:
    return 0.90 * metrics["f1"] + 0.10 * metrics["acc"] - 0.05 * metrics["ece"]


## Data structures
Dataset wrappers and typed containers used across preprocessing and training.

In [ ]:
# ============================================================
# Data structures
# ============================================================

class WindowDataset(Dataset):
    def __init__(self, windows: np.ndarray, labels: np.ndarray):
        assert len(windows) == len(labels)
        self.x = torch.from_numpy(np.ascontiguousarray(windows)).float()
        self.y = torch.from_numpy(np.ascontiguousarray(labels)).long()

    def __len__(self) -> int:
        return len(self.y)

    def __getitem__(self, idx: int):
        return self.x[idx], self.y[idx]


@dataclass
class SubjectPartition:
    windows: np.ndarray
    labels: np.ndarray


@dataclass
class FoldData:
    heldout_id: str
    train_ids: List[str]
    train_subjects: Dict[str, Dict[str, SubjectPartition]]
    heldout_subject: Dict[str, SubjectPartition]
    num_classes: int
    feature_dim: int
    label_map: Dict[int, int]


## Loading and preprocessing
Data discovery, cleaning, segmentation, leakage-safe splitting, scaling, and loader construction.

In [ ]:
# ============================================================
# Loading and preprocessing
# ============================================================

def discover_subject_files(cfg: Config) -> Dict[str, Path]:
    paths = sorted(Path(cfg.data_dir).glob(cfg.subject_glob))
    if not paths:
        raise FileNotFoundError(
            f"No PAMAP2 files found in {cfg.data_dir!r} with glob {cfg.subject_glob!r}."
        )
    return {p.stem: p for p in paths}


def load_subject_dataframe(file_path: Path, cfg: Config) -> pd.DataFrame:
    cols = pamap2_columns()
    df = pd.read_csv(file_path, sep=r"\s+", header=None, names=cols, engine="python")

    # only keep valid activities
    df = df[df["activity_id"].isin(cfg.valid_activity_ids)].copy()
    if df.empty:
        raise RuntimeError(f"{file_path.name} has no rows for valid PAMAP2 activity IDs.")

    # PAMAP2 uses -1 as missing marker
    df.replace(-1.0, np.nan, inplace=True)

    feat_cols = feature_columns()
    df[feat_cols] = df[feat_cols].interpolate(method="linear", limit_direction="both", axis=0)
    df[feat_cols] = df[feat_cols].ffill().bfill()
    df[feat_cols] = df[feat_cols].fillna(df[feat_cols].median())

    return df.reset_index(drop=True)


def build_global_label_map(subject_dfs: Dict[str, pd.DataFrame]) -> Dict[int, int]:
    activity_ids = sorted({int(a) for df in subject_dfs.values() for a in df["activity_id"].unique()})
    return {aid: idx for idx, aid in enumerate(activity_ids)}


def encode_subject(
    df: pd.DataFrame,
    label_map: Dict[int, int],
) -> Tuple[np.ndarray, np.ndarray]:
    X = df[feature_columns()].to_numpy(dtype=np.float32)
    y = df["activity_id"].map(label_map).to_numpy(dtype=np.int64)
    return X, y


def constant_label_segments(y: np.ndarray) -> List[Tuple[int, int, int]]:
    segments: List[Tuple[int, int, int]] = []
    if len(y) == 0:
        return segments

    s = 0
    n = len(y)
    while s < n:
        e = s + 1
        while e < n and y[e] == y[s]:
            e += 1
        segments.append((s, e, int(y[s])))
        s = e
    return segments


def generate_constant_label_windows(
    X: np.ndarray,
    y: np.ndarray,
    seq_len: int,
    stride: int,
    drop_mixed_windows: bool = True,
) -> Tuple[np.ndarray, np.ndarray]:
    windows: List[np.ndarray] = []
    labels: List[int] = []

    if len(X) == 0:
        feat_dim = X.shape[1] if X.ndim == 2 else 0
        return np.empty((0, seq_len, feat_dim), dtype=np.float32), np.empty((0,), dtype=np.int64)

    s = 0
    n = len(y)
    while s < n:
        e = s + 1
        while e < n and y[e] == y[s]:
            e += 1

        seg_x = X[s:e]
        seg_y = y[s]
        seg_len = len(seg_x)

        if seg_len >= seq_len:
            for start in range(0, seg_len - seq_len + 1, stride):
                win_x = seg_x[start : start + seq_len]
                if drop_mixed_windows and not np.all(y[s + start : s + start + seq_len] == seg_y):
                    continue
                windows.append(win_x)
                labels.append(int(seg_y))
        s = e

    feat_dim = X.shape[1]
    if not windows:
        return np.empty((0, seq_len, feat_dim), dtype=np.float32), np.empty((0,), dtype=np.int64)
    return np.stack(windows).astype(np.float32), np.array(labels, dtype=np.int64)


def empty_partition(seq_len: int, feat_dim: int) -> SubjectPartition:
    return SubjectPartition(
        windows=np.empty((0, seq_len, feat_dim), dtype=np.float32),
        labels=np.empty((0,), dtype=np.int64),
    )


def concat_partitions(parts: List[SubjectPartition], seq_len: int, feat_dim: int) -> SubjectPartition:
    non_empty = [p for p in parts if len(p.labels) > 0]
    if not non_empty:
        return empty_partition(seq_len, feat_dim)
    return SubjectPartition(
        windows=np.concatenate([p.windows for p in non_empty], axis=0).astype(np.float32),
        labels=np.concatenate([p.labels for p in non_empty], axis=0).astype(np.int64),
    )


def labels_present(labels: np.ndarray) -> List[int]:
    if len(labels) == 0:
        return []
    return np.where(np.bincount(labels) > 0)[0].tolist()


def label_histogram(labels: np.ndarray, num_classes: int) -> Dict[int, int]:
    if len(labels) == 0:
        return {}
    counts = np.bincount(labels, minlength=num_classes)
    return {i: int(c) for i, c in enumerate(counts) if c > 0}


def allocate_lengths_with_minimum(
    total_len: int,
    ratios: List[float],
    min_lengths: List[int],
) -> Optional[List[int]]:
    """
    Split total_len into k parts:
    - each part >= min_lengths[i]
    - remaining budget allocated by ratios
    """
    min_sum = int(sum(min_lengths))
    if total_len < min_sum:
        return None

    ratios_arr = np.array(ratios, dtype=np.float64)
    if ratios_arr.sum() <= 0:
        ratios_arr = np.ones_like(ratios_arr)
    ratios_arr = ratios_arr / ratios_arr.sum()

    extra_total = int(total_len - min_sum)
    raw_extra = ratios_arr * extra_total
    extra = np.floor(raw_extra).astype(np.int64)

    remainder = extra_total - int(extra.sum())
    if remainder > 0:
        frac = raw_extra - extra
        order = np.argsort(-frac)
        for idx in order[:remainder]:
            extra[idx] += 1

    out = (np.array(min_lengths, dtype=np.int64) + extra).astype(np.int64)
    return [int(v) for v in out]


def split_single_segment_train_val(
    seg_len: int,
    cfg: Config,
) -> Dict[str, List[Tuple[int, int]]]:
    """
    Split ONE constant-label segment into train/val subranges with a purge gap,
    so train and val both contain the class whenever the segment is long enough.
    """
    gap = cfg.purge_gap_raw
    min_win = cfg.seq_len

    # Need room for train + gap + val
    if seg_len < (2 * min_win + gap):
        return {
            "train": [(0, seg_len)] if seg_len >= min_win else [],
            "val": [],
        }

    usable = seg_len - gap
    lengths = allocate_lengths_with_minimum(
        total_len=usable,
        ratios=[cfg.train_subject_train_ratio, cfg.train_subject_val_ratio],
        min_lengths=[min_win, min_win],
    )
    if lengths is None:
        return {
            "train": [(0, seg_len)] if seg_len >= min_win else [],
            "val": [],
        }

    train_len, val_len = lengths
    train_range = (0, train_len)
    val_range = (train_len + gap, train_len + gap + val_len)

    return {
        "train": [train_range],
        "val": [val_range],
    }


def split_single_segment_support_adapt_test(
    seg_len: int,
    cfg: Config,
) -> Dict[str, List[Tuple[int, int]]]:
    """
    Split ONE constant-label segment into support/adapt_val/test subranges with purge gaps,
    so all three held-out partitions contain the class whenever the segment is long enough.
    """
    gap = cfg.purge_gap_raw
    min_win = cfg.seq_len

    # Need room for support + gap + adapt + gap + test
    if seg_len < (3 * min_win + 2 * gap):
        # fallback hierarchy: if not enough for 3-way, try 2-way, else put all in test
        if seg_len >= (2 * min_win + gap):
            usable = seg_len - gap
            lengths = allocate_lengths_with_minimum(
                total_len=usable,
                ratios=[cfg.heldout_support_ratio + cfg.heldout_adapt_val_ratio, cfg.heldout_test_ratio],
                min_lengths=[min_win, min_win],
            )
            if lengths is None:
                return {
                    "support": [],
                    "adapt_val": [],
                    "test": [(0, seg_len)] if seg_len >= min_win else [],
                }

            sa_len, test_len = lengths

            # Try splitting the first block into support + adapt_val too
            if sa_len >= (2 * min_win + gap):
                sa_usable = sa_len - gap
                sa_lengths = allocate_lengths_with_minimum(
                    total_len=sa_usable,
                    ratios=[cfg.heldout_support_ratio, cfg.heldout_adapt_val_ratio],
                    min_lengths=[min_win, min_win],
                )
                if sa_lengths is not None:
                    support_len, adapt_len = sa_lengths
                    return {
                        "support": [(0, support_len)],
                        "adapt_val": [(support_len + gap, support_len + gap + adapt_len)],
                        "test": [(sa_len + gap, sa_len + gap + test_len)],
                    }

            # If still impossible, use support + test and leave adapt empty
            return {
                "support": [(0, sa_len)],
                "adapt_val": [],
                "test": [(sa_len + gap, sa_len + gap + test_len)],
            }

        return {
            "support": [],
            "adapt_val": [],
            "test": [(0, seg_len)] if seg_len >= min_win else [],
        }

    usable = seg_len - 2 * gap
    lengths = allocate_lengths_with_minimum(
        total_len=usable,
        ratios=[cfg.heldout_support_ratio, cfg.heldout_adapt_val_ratio, cfg.heldout_test_ratio],
        min_lengths=[min_win, min_win, min_win],
    )
    if lengths is None:
        return {
            "support": [],
            "adapt_val": [],
            "test": [(0, seg_len)] if seg_len >= min_win else [],
        }

    support_len, adapt_len, test_len = lengths

    support_range = (0, support_len)
    adapt_range = (support_len + gap, support_len + gap + adapt_len)
    test_range = (support_len + gap + adapt_len + gap, support_len + gap + adapt_len + gap + test_len)

    return {
        "support": [support_range],
        "adapt_val": [adapt_range],
        "test": [test_range],
    }


def windows_from_absolute_ranges(
    X: np.ndarray,
    y: np.ndarray,
    abs_ranges: List[Tuple[int, int]],
    cfg: Config,
) -> SubjectPartition:
    feat_dim = X.shape[1]
    parts: List[SubjectPartition] = []

    for a, b in abs_ranges:
        if (b - a) < cfg.seq_len:
            continue
        w, lab = generate_constant_label_windows(
            X[a:b],
            y[a:b],
            seq_len=cfg.seq_len,
            stride=cfg.stride,
            drop_mixed_windows=cfg.drop_mixed_windows,
        )
        if len(lab) > 0:
            parts.append(SubjectPartition(windows=w, labels=lab))

    return concat_partitions(parts, cfg.seq_len, feat_dim)


def build_subject_partitions_train(
    X: np.ndarray,
    y: np.ndarray,
    cfg: Config,
) -> Dict[str, SubjectPartition]:
    feat_dim = X.shape[1]
    train_parts: List[SubjectPartition] = []
    val_parts: List[SubjectPartition] = []

    segments = constant_label_segments(y)

    for s, e, _lab in segments:
        seg_len = e - s
        rel = split_single_segment_train_val(seg_len, cfg)

        train_abs = [(s + a, s + b) for a, b in rel["train"]]
        val_abs = [(s + a, s + b) for a, b in rel["val"]]

        train_parts.append(windows_from_absolute_ranges(X, y, train_abs, cfg))
        val_parts.append(windows_from_absolute_ranges(X, y, val_abs, cfg))

    return {
        "train": concat_partitions(train_parts, cfg.seq_len, feat_dim),
        "val": concat_partitions(val_parts, cfg.seq_len, feat_dim),
    }


def build_subject_partitions_heldout(
    X: np.ndarray,
    y: np.ndarray,
    cfg: Config,
) -> Dict[str, SubjectPartition]:
    feat_dim = X.shape[1]
    support_parts: List[SubjectPartition] = []
    adapt_parts: List[SubjectPartition] = []
    test_parts: List[SubjectPartition] = []

    segments = constant_label_segments(y)

    for s, e, _lab in segments:
        seg_len = e - s
        rel = split_single_segment_support_adapt_test(seg_len, cfg)

        support_abs = [(s + a, s + b) for a, b in rel["support"]]
        adapt_abs = [(s + a, s + b) for a, b in rel["adapt_val"]]
        test_abs = [(s + a, s + b) for a, b in rel["test"]]

        support_parts.append(windows_from_absolute_ranges(X, y, support_abs, cfg))
        adapt_parts.append(windows_from_absolute_ranges(X, y, adapt_abs, cfg))
        test_parts.append(windows_from_absolute_ranges(X, y, test_abs, cfg))

    return {
        "support": concat_partitions(support_parts, cfg.seq_len, feat_dim),
        "adapt_val": concat_partitions(adapt_parts, cfg.seq_len, feat_dim),
        "test": concat_partitions(test_parts, cfg.seq_len, feat_dim),
    }


def print_partition_debug(
    sid: str,
    packaged: Dict[str, SubjectPartition],
    num_classes: int,
    heldout: bool,
) -> None:
    if heldout:
        split_order = ["support", "adapt_val", "test"]
    else:
        split_order = ["train", "val"]

    print(f"[PARTITIONS][{sid}]")
    for split_name in split_order:
        part = packaged[split_name]
        print(
            f"  {split_name}: windows={len(part.labels)}, "
            f"classes={labels_present(part.labels)}, "
            f"hist={label_histogram(part.labels, num_classes)}"
        )

    if not heldout:
        train_set = set(labels_present(packaged["train"].labels))
        val_set = set(labels_present(packaged["val"].labels))
        missing = sorted(val_set - train_set)
        print(f"  train/val overlap ok? missing_val_in_train={missing}")
    else:
        support_set = set(labels_present(packaged["support"].labels))
        adapt_set = set(labels_present(packaged["adapt_val"].labels))
        test_set = set(labels_present(packaged["test"].labels))
        print(f"  support∩adapt classes={sorted(support_set & adapt_set)}")
        print(f"  support∩test classes={sorted(support_set & test_set)}")
        print(f"  adapt∩test classes={sorted(adapt_set & test_set)}")


def build_fold_data(
    subject_arrays: Dict[str, Tuple[np.ndarray, np.ndarray]],
    heldout_id: str,
    label_map: Dict[int, int],
    cfg: Config,
) -> FoldData:
    train_ids = [sid for sid in subject_arrays.keys() if sid != heldout_id]

    train_subjects: Dict[str, Dict[str, SubjectPartition]] = {}
    heldout_subject: Dict[str, SubjectPartition] = {}

    num_classes = len(label_map)

    for sid, (X, y) in subject_arrays.items():
        if sid == heldout_id:
            packaged = build_subject_partitions_heldout(X, y, cfg)
            heldout_subject = packaged
            print_partition_debug(sid, packaged, num_classes, heldout=True)
        else:
            packaged = build_subject_partitions_train(X, y, cfg)
            train_subjects[sid] = packaged
            print_partition_debug(sid, packaged, num_classes, heldout=False)

    scaler = StandardScaler()
    any_train = False
    for sid in train_ids:
        w = train_subjects[sid]["train"].windows
        if len(w) > 0:
            scaler.partial_fit(w.reshape(-1, w.shape[-1]))
            any_train = True
    if not any_train:
        raise RuntimeError("No training windows found across non-heldout subjects.")

    def transform_partition(part: SubjectPartition) -> SubjectPartition:
        if len(part.windows) == 0:
            return part
        shape = part.windows.shape
        flat = part.windows.reshape(-1, shape[-1])
        flat = scaler.transform(flat)
        return SubjectPartition(
            windows=flat.reshape(shape).astype(np.float32),
            labels=part.labels.astype(np.int64),
        )

    for sid in train_subjects.keys():
        for split_name in train_subjects[sid].keys():
            train_subjects[sid][split_name] = transform_partition(train_subjects[sid][split_name])

    for split_name in heldout_subject.keys():
        heldout_subject[split_name] = transform_partition(heldout_subject[split_name])

    for sid in train_ids:
        if len(train_subjects[sid]["train"].windows) < cfg.min_windows_per_partition:
            raise RuntimeError(f"{sid} has too few train windows ({len(train_subjects[sid]['train'].windows)}).")
        if len(train_subjects[sid]["val"].windows) < max(1, cfg.min_windows_per_partition // 2):
            raise RuntimeError(f"{sid} has too few val windows ({len(train_subjects[sid]['val'].windows)}).")

    for split_name in ("support", "adapt_val", "test"):
        if len(heldout_subject[split_name].windows) < max(1, cfg.min_windows_per_partition // 2):
            raise RuntimeError(
                f"Held-out {heldout_id} split {split_name} has too few windows "
                f"({len(heldout_subject[split_name].windows)})."
            )

    feat_dim = next(iter(subject_arrays.values()))[0].shape[1]
    return FoldData(
        heldout_id=heldout_id,
        train_ids=train_ids,
        train_subjects=train_subjects,
        heldout_subject=heldout_subject,
        num_classes=num_classes,
        feature_dim=feat_dim,
        label_map=label_map,
    )


def build_fold_summary(fold: FoldData) -> Dict[str, Dict[str, int]]:
    out: Dict[str, Dict[str, int]] = {}
    for sid in fold.train_ids:
        out[sid] = {
            "train": int(len(fold.train_subjects[sid]["train"].labels)),
            "val": int(len(fold.train_subjects[sid]["val"].labels)),
        }
    out[fold.heldout_id] = {
        "support": int(len(fold.heldout_subject["support"].labels)),
        "adapt_val": int(len(fold.heldout_subject["adapt_val"].labels)),
        "test": int(len(fold.heldout_subject["test"].labels)),
    }
    return out


def make_loader(
    part: SubjectPartition,
    batch_size: int,
    shuffle: bool,
    cfg: Config,
) -> DataLoader:
    ds = WindowDataset(part.windows, part.labels)
    loader_kwargs = dict(
        batch_size=batch_size,
        shuffle=shuffle,
        num_workers=cfg.num_workers,
        pin_memory=(cfg.pin_memory and str(cfg.device).startswith("cuda")),
        persistent_workers=(cfg.num_workers > 0),
        drop_last=False,
    )
    return DataLoader(ds, **loader_kwargs)


def fold_to_loaders(
    fold: FoldData,
    cfg: Config,
) -> Dict[str, object]:
    loaders: Dict[str, object] = {
        "train_clients": {},
        "heldout": {},
        "num_classes": fold.num_classes,
        "feature_dim": fold.feature_dim,
        "heldout_id": fold.heldout_id,
        "train_ids": fold.train_ids,
    }

    for sid in fold.train_ids:
        loaders["train_clients"][sid] = {
            "train": make_loader(fold.train_subjects[sid]["train"], cfg.batch_size, True, cfg),
            "val": make_loader(fold.train_subjects[sid]["val"], cfg.batch_size, False, cfg),
            "n_train": len(fold.train_subjects[sid]["train"].labels),
        }

    for split_name in ("support", "adapt_val", "test"):
        bs = cfg.personal_batch_size if split_name != "test" else cfg.batch_size
        loaders["heldout"][split_name] = make_loader(
            fold.heldout_subject[split_name], bs, split_name == "support", cfg
        )

    loaders["summary"] = build_fold_summary(fold)
    return loaders


## Model
Temporal encoder architecture including temporal stem, positional encoding, Transformer layers, and attentive pooling.

In [ ]:
# ============================================================
# Model
# ============================================================

class SinusoidalPositionalEncoding(nn.Module):
    def __init__(self, d_model: int, max_len: int = 2048):
        super().__init__()
        pe = torch.zeros(max_len, d_model, dtype=torch.float32)
        pos = torch.arange(0, max_len, dtype=torch.float32).unsqueeze(1)
        div = torch.exp(torch.arange(0, d_model, 2, dtype=torch.float32) * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(pos * div)
        pe[:, 1::2] = torch.cos(pos * div)
        self.register_buffer("pe", pe.unsqueeze(0), persistent=False)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return x + self.pe[:, : x.size(1)]


class TemporalStem(nn.Module):
    def __init__(self, in_dim: int, d_model: int, dropout: float):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv1d(in_dim, d_model, kernel_size=5, padding=2, bias=False),
            nn.BatchNorm1d(d_model),
            nn.GELU(),
            nn.Conv1d(d_model, d_model, kernel_size=3, padding=1, groups=d_model, bias=False),
            nn.BatchNorm1d(d_model),
            nn.GELU(),
            nn.Conv1d(d_model, d_model, kernel_size=1, bias=False),
            nn.BatchNorm1d(d_model),
            nn.GELU(),
            nn.Dropout(dropout),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = x.transpose(1, 2)
        x = self.net(x)
        return x.transpose(1, 2)


class AttentivePool(nn.Module):
    def __init__(self, d_model: int):
        super().__init__()
        self.score = nn.Linear(d_model, 1)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        attn = torch.softmax(self.score(x).squeeze(-1), dim=1)
        pooled = torch.sum(attn.unsqueeze(-1) * x, dim=1)
        return pooled


class TemporalEncoder(nn.Module):
    def __init__(self, in_dim: int, cfg: Config):
        super().__init__()
        self.stem = TemporalStem(in_dim, cfg.d_model, cfg.dropout)
        self.pos = SinusoidalPositionalEncoding(cfg.d_model, max_len=max(2048, cfg.seq_len + 8))

        enc_layer = nn.TransformerEncoderLayer(
            d_model=cfg.d_model,
            nhead=cfg.n_heads,
            dim_feedforward=cfg.ff_dim,
            dropout=cfg.dropout,
            batch_first=True,
            norm_first=False,
            activation="gelu",
        )
        self.encoder = nn.TransformerEncoder(enc_layer, num_layers=cfg.n_layers)
        self.pool = AttentivePool(cfg.d_model)
        self.out_norm = nn.LayerNorm(cfg.d_model)
        self.proj = nn.Linear(cfg.d_model, cfg.emb_dim)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        h = self.stem(x)
        h = self.pos(h)
        h = self.encoder(h)
        h = self.pool(h)
        h = self.out_norm(h)
        z = self.proj(h)
        z = F.normalize(z, dim=-1)
        return z


## Prototype utilities
Prototype construction, logits, local objective, and class weighting.

In [ ]:
# ============================================================
# Prototype utilities
# ============================================================

def orthonormal_random(d: int, r: int, device: torch.device) -> torch.Tensor:
    q, _ = torch.linalg.qr(torch.randn(d, r, device=device))
    return q[:, :r].contiguous()


def personalized_prototypes_ccd(
    global_proto: torch.Tensor,
    proto_basis: torch.Tensor,
    client_A: torch.Tensor,
) -> torch.Tensor:
    proto = global_proto + client_A @ proto_basis.T
    proto = F.normalize(proto, dim=-1)
    return proto


def proto_logits(
    z: torch.Tensor,
    global_proto: torch.Tensor,
    proto_basis: torch.Tensor,
    client_A: torch.Tensor,
    log_tau: torch.Tensor,
    cfg: Config,
) -> torch.Tensor:
    P_i = personalized_prototypes_ccd(global_proto, proto_basis, client_A)
    tau = torch.exp(log_tau).clamp(cfg.tau_min, cfg.tau_max)
    return tau * (z @ P_i.T)


def local_loss(
    z: torch.Tensor,
    y: torch.Tensor,
    global_proto: torch.Tensor,
    proto_basis: torch.Tensor,
    client_A: torch.Tensor,
    log_tau: torch.Tensor,
    cfg: Config,
    class_weight: Optional[torch.Tensor] = None,
) -> Tuple[torch.Tensor, Dict[str, float]]:
    P_i = personalized_prototypes_ccd(global_proto, proto_basis, client_A)
    logits = proto_logits(z, global_proto, proto_basis, client_A, log_tau, cfg)

    loss_proto = F.cross_entropy(
        logits,
        y,
        weight=class_weight,
        label_smoothing=0.05,
    )
    align = 1.0 - torch.sum(z * P_i[y], dim=1).mean()
    reg_A = client_A.pow(2).mean()
    reg_tau = (log_tau - math.log(cfg.tau_init)) ** 2

    loss = (
        loss_proto
        + cfg.lambda_align * align
        + cfg.lambda_A * reg_A
        + cfg.lambda_tau * reg_tau
    )

    stats = {
        "loss_proto": float(loss_proto.detach().item()),
        "loss_align": float(align.detach().item()),
        "loss_regA": float(reg_A.detach().item()),
        "loss_regTau": float(reg_tau.detach().item()),
        "tau": float(torch.exp(log_tau.detach()).clamp(cfg.tau_min, cfg.tau_max).item()),
    }
    return loss, stats


def make_class_weight_from_loader(
    train_loader: DataLoader,
    num_classes: int,
    device: torch.device,
    cfg: Config,
) -> torch.Tensor:
    labels = train_loader.dataset.y.detach().cpu().numpy()
    counts = np.bincount(labels, minlength=num_classes).astype(np.float32)

    present = counts > 0
    weights = np.ones(num_classes, dtype=np.float32)

    if present.any():
        ref = float(np.median(counts[present]))
        weights[present] = np.power(ref / np.clip(counts[present], 1.0, None), cfg.class_weight_power)
        weights[present] = np.clip(weights[present], 1.0 / cfg.class_weight_max, cfg.class_weight_max)
        weights[present] /= max(weights[present].mean(), 1e-8)

    return torch.tensor(weights, dtype=torch.float32, device=device)


## Evaluation
Prediction and evaluation helpers used during validation and test time.

In [ ]:
# ============================================================
# Evaluation
# ============================================================

@torch.inference_mode()
def predict_loader(
    model: nn.Module,
    loader: DataLoader,
    global_proto: torch.Tensor,
    proto_basis: torch.Tensor,
    client_A: torch.Tensor,
    log_tau: torch.Tensor,
    device: torch.device,
    cfg: Config,
    use_amp: bool = True,
) -> Tuple[np.ndarray, np.ndarray]:
    model.eval()
    probs_all: List[np.ndarray] = []
    y_all: List[np.ndarray] = []

    autocast_device = "cuda" if device.type == "cuda" else "cpu"
    amp_enabled = use_amp and device.type == "cuda"

    for xb, yb in loader:
        xb = xb.to(device, non_blocking=True)
        yb = yb.to(device, non_blocking=True)
        with torch.amp.autocast(autocast_device, enabled=amp_enabled):
            z = model(xb)
            logits = proto_logits(z, global_proto, proto_basis, client_A, log_tau, cfg)
            probs = torch.softmax(logits, dim=1)
        probs_all.append(probs.detach().cpu().numpy())
        y_all.append(yb.detach().cpu().numpy())

    if not probs_all:
        num_classes = global_proto.shape[0]
        return np.empty((0, num_classes), dtype=np.float32), np.empty((0,), dtype=np.int64)

    return np.concatenate(probs_all, axis=0), np.concatenate(y_all, axis=0)


def evaluate_loader(
    model: nn.Module,
    loader: DataLoader,
    global_proto: torch.Tensor,
    proto_basis: torch.Tensor,
    client_A: torch.Tensor,
    log_tau: torch.Tensor,
    cfg: Config,
    device: torch.device,
) -> Dict[str, float]:
    probs, labels = predict_loader(
        model, loader, global_proto, proto_basis, client_A, log_tau, device, cfg, cfg.amp
    )
    return summarize_probs(probs, labels, num_classes=global_proto.shape[0], ece_bins=cfg.ece_bins)


## Client training and prototype extraction
Local client optimization and confidence-weighted empirical prototype extraction.

In [ ]:
# ============================================================
# Client training and prototype extraction
# ============================================================

def clone_model(model: nn.Module) -> nn.Module:
    return copy.deepcopy(model)


def train_one_client(
    global_model: nn.Module,
    train_loader: DataLoader,
    val_loader: DataLoader,
    global_proto: torch.Tensor,
    proto_basis: torch.Tensor,
    A_init: torch.Tensor,
    log_tau_init: torch.Tensor,
    cfg: Config,
    device: torch.device,
    warmup_extract: bool = False,
) -> Tuple[Dict[str, torch.Tensor], torch.Tensor, torch.Tensor, Dict[str, torch.Tensor]]:
    del val_loader  # local validation was unused by the outer loop and only added runtime

    local_model = clone_model(global_model).to(device)

    client_A = nn.Parameter(A_init.clone().to(device))
    log_tau = nn.Parameter(log_tau_init.clone().to(device))
    class_weight = make_class_weight_from_loader(train_loader, global_proto.shape[0], device, cfg)

    optimizer = torch.optim.AdamW(
        list(local_model.parameters()) + [client_A, log_tau],
        lr=cfg.lr_encoder,
        weight_decay=cfg.weight_decay,
    )
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=max(1, cfg.local_epochs))
    scaler = torch.amp.GradScaler(enabled=(cfg.amp and device.type == "cuda"))

    autocast_device = "cuda" if device.type == "cuda" else "cpu"
    amp_enabled = cfg.amp and device.type == "cuda"

    optimizer_steps = 0
    scheduler_steps = 0

    for _epoch in range(cfg.local_epochs):
        local_model.train()

        for xb, yb in train_loader:
            xb = xb.to(device, non_blocking=True)
            yb = yb.to(device, non_blocking=True)

            optimizer.zero_grad(set_to_none=True)

            with torch.amp.autocast(autocast_device, enabled=amp_enabled):
                z = local_model(xb)
                loss, _ = local_loss(
                    z, yb, global_proto, proto_basis, client_A, log_tau, cfg, class_weight=class_weight
                )

            if amp_enabled:
                scaler.scale(loss).backward()

                if cfg.grad_clip is not None and cfg.grad_clip > 0:
                    scaler.unscale_(optimizer)
                    torch.nn.utils.clip_grad_norm_(
                        list(local_model.parameters()) + [client_A, log_tau],
                        cfg.grad_clip,
                    )

                old_scale = scaler.get_scale()
                scaler.step(optimizer)
                scaler.update()
                new_scale = scaler.get_scale()

                if new_scale >= old_scale:
                    optimizer_steps += 1
            else:
                loss.backward()
                if cfg.grad_clip is not None and cfg.grad_clip > 0:
                    torch.nn.utils.clip_grad_norm_(
                        list(local_model.parameters()) + [client_A, log_tau],
                        cfg.grad_clip,
                    )
                optimizer.step()
                optimizer_steps += 1

        if optimizer_steps > scheduler_steps:
            scheduler.step()
            scheduler_steps += 1

    proto_summary = {}
    emp, present, counts = extract_confidence_weighted_prototypes(
        model=local_model,
        loader=train_loader,
        global_proto=global_proto,
        proto_basis=proto_basis,
        client_A=client_A.detach(),
        log_tau=log_tau.detach(),
        num_classes=global_proto.shape[0],
        emb_dim=cfg.emb_dim,
        device=device,
        cfg=cfg,
        use_amp=cfg.amp,
        warmup=warmup_extract,
    )
    proto_summary["empirical"] = emp
    proto_summary["present"] = present
    proto_summary["counts"] = counts

    state_dict = {k: v.detach().cpu().clone() for k, v in local_model.state_dict().items()}

    del local_model

    return (
        state_dict,
        client_A.detach().cpu().clone(),
        log_tau.detach().cpu().clone(),
        proto_summary,
    )


@torch.inference_mode()
def extract_confidence_weighted_prototypes(
    model: nn.Module,
    loader: DataLoader,
    global_proto: torch.Tensor,
    proto_basis: torch.Tensor,
    client_A: torch.Tensor,
    log_tau: torch.Tensor,
    num_classes: int,
    emb_dim: int,
    device: torch.device,
    cfg: Config,
    use_amp: bool = True,
    warmup: bool = False,
) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
    model.eval()

    proto_sum = torch.zeros(num_classes, emb_dim, device=device)
    weight_sum = torch.zeros(num_classes, device=device)
    counts = torch.zeros(num_classes, device=device)

    autocast_device = "cuda" if device.type == "cuda" else "cpu"
    amp_enabled = use_amp and device.type == "cuda"

    for xb, yb in loader:
        xb = xb.to(device, non_blocking=True)
        yb = yb.to(device, non_blocking=True)

        with torch.amp.autocast(autocast_device, enabled=amp_enabled):
            z = model(xb)
            logits = proto_logits(z, global_proto, proto_basis, client_A, log_tau, cfg)
            probs = torch.softmax(logits, dim=1)

        if warmup:
            weights = torch.ones_like(yb, dtype=z.dtype, device=device)
        else:
            true_conf = probs.gather(1, yb.unsqueeze(1)).squeeze(1)
            max_conf = probs.max(dim=1).values
            weights = 0.7 * true_conf + 0.3 * max_conf

        proto_sum.index_add_(0, yb, z * weights.unsqueeze(1))
        weight_sum.index_add_(0, yb, weights)
        counts.index_add_(0, yb, torch.ones_like(weights))

    empirical = torch.zeros_like(proto_sum)
    present = (weight_sum > 0) & (counts >= cfg.min_proto_samples_per_class)

    empirical[present] = proto_sum[present] / weight_sum[present].unsqueeze(1)
    empirical[present] = F.normalize(empirical[present], dim=1)

    return empirical.detach().cpu(), present.detach().cpu(), counts.detach().cpu()


## Server aggregation and geometry update
Federated averaging and server-side low-rank geometry updates.

In [ ]:
# ============================================================
# Server aggregation and geometry update
# ============================================================

def average_state_dicts(
    state_dicts: List[Dict[str, torch.Tensor]],
    weights: List[float],
) -> Dict[str, torch.Tensor]:
    total = float(sum(weights))
    weights = [w / total for w in weights]
    out: Dict[str, torch.Tensor] = {}
    keys = state_dicts[0].keys()
    for k in keys:
        ref = state_dicts[0][k]
        if torch.is_floating_point(ref):
            acc = None
            for sd, w in zip(state_dicts, weights):
                tensor = sd[k].float()
                acc = tensor * w if acc is None else acc + tensor * w
            out[k] = acc.to(dtype=ref.dtype)
        else:
            out[k] = ref.clone()
    return out


def solve_A_closed_form(
    empirical_proto: torch.Tensor,
    present: torch.Tensor,
    global_proto: torch.Tensor,
    proto_basis: torch.Tensor,
    lambda_A: float,
) -> torch.Tensor:
    BtB = proto_basis.T @ proto_basis
    r = BtB.shape[0]
    inv = torch.linalg.inv(BtB + lambda_A * torch.eye(r, device=proto_basis.device, dtype=proto_basis.dtype))
    A = (empirical_proto - global_proto) @ proto_basis @ inv
    A = A * present.float().unsqueeze(1)
    return A


def server_geometry_update(
    global_proto: torch.Tensor,
    proto_basis: torch.Tensor,
    proto_summaries: Dict[str, Dict[str, torch.Tensor]],
    cfg: Config,
    device: torch.device,
) -> Tuple[torch.Tensor, torch.Tensor, Dict[str, torch.Tensor]]:
    """
    Minimizes approximately:
        sum_{i,c} w_{i,c} || H_ic - p_c - B a_ic ||^2
        + lambda_A sum_i ||A_i||^2 + lambda_B ||B^T B - I||^2
    """
    old_P = global_proto.clone().to(device)
    old_B = proto_basis.clone().to(device)

    P = old_P.clone()
    B = old_B.clone()

    client_ids = list(proto_summaries.keys())
    H = torch.stack([proto_summaries[sid]["empirical"].to(device) for sid in client_ids], dim=0)
    M = torch.stack([proto_summaries[sid]["present"].to(device) for sid in client_ids], dim=0).bool()
    W = torch.stack([proto_summaries[sid]["counts"].to(device) for sid in client_ids], dim=0).float()
    W = torch.where(M, torch.sqrt(torch.clamp(W, min=0.0)), torch.zeros_like(W))

    K, C, d = H.shape
    r = B.shape[1]

    A_list = []
    for i in range(K):
        A_i = solve_A_closed_form(H[i], M[i], P, B, cfg.lambda_A)
        A_list.append(A_i)
    A = torch.stack(A_list, dim=0)

    for _ in range(cfg.server_alt_iters):
        numer = torch.zeros_like(P)
        denom = torch.zeros(C, device=device)
        for i in range(K):
            recon_free = H[i] - A[i] @ B.T
            numer += W[i].unsqueeze(1) * recon_free
            denom += W[i]

        keep = denom > 0
        P[keep] = numer[keep] / denom[keep].unsqueeze(1)
        P = F.normalize(P, dim=1)

        B_param = nn.Parameter(B.clone())
        opt = torch.optim.Adam([B_param], lr=cfg.server_geom_lr)

        for _step in range(cfg.server_geom_steps):
            opt.zero_grad(set_to_none=True)
            recon = P.unsqueeze(0) + torch.matmul(A, B_param.T)
            sq = ((H - recon) ** 2).sum(dim=2)
            data_term = (W * sq).sum() / (W.sum() + 1e-8)
            orth = ((B_param.T @ B_param) - torch.eye(r, device=device)).pow(2).mean()
            loss = data_term + cfg.lambda_B * orth
            loss.backward()
            opt.step()

        with torch.no_grad():
            q, _ = torch.linalg.qr(B_param.data)
            B = q[:, :r].contiguous()

        A_list = []
        for i in range(K):
            A_i = solve_A_closed_form(H[i], M[i], P, B, cfg.lambda_A)
            A_list.append(A_i)
        A = torch.stack(A_list, dim=0)

    # momentum smoothing of shared geometry
    P = F.normalize(cfg.server_proto_momentum * old_P + (1.0 - cfg.server_proto_momentum) * P, dim=1)
    B_blend = cfg.server_basis_momentum * old_B + (1.0 - cfg.server_basis_momentum) * B
    q, _ = torch.linalg.qr(B_blend)
    B = q[:, :r].contiguous()

    A_list = []
    for i in range(K):
        A_i = solve_A_closed_form(H[i], M[i], P, B, cfg.lambda_A)
        A_list.append(A_i)
    A = torch.stack(A_list, dim=0)

    client_A = {sid: A[idx].detach().cpu().clone() for idx, sid in enumerate(client_ids)}
    return P.detach(), B.detach(), client_A


## Held-out personalization
Adaptation procedure for the unseen held-out subject.

In [ ]:
# ============================================================
# Held-out personalization
# ============================================================

def adapt_heldout_client(
    model: nn.Module,
    support_loader: DataLoader,
    adapt_val_loader: DataLoader,
    global_proto: torch.Tensor,
    proto_basis: torch.Tensor,
    cfg: Config,
    device: torch.device,
    init_log_tau: Optional[torch.Tensor] = None,
) -> Tuple[torch.Tensor, torch.Tensor, Dict[str, float], Dict[str, object]]:
    frozen_model = clone_model(model).to(device)
    frozen_model.eval()
    for p in frozen_model.parameters():
        p.requires_grad = False

    C, d = global_proto.shape
    r = proto_basis.shape[1]
    global_proto = global_proto.to(device)
    proto_basis = proto_basis.to(device)

    if init_log_tau is None:
        init_log_tau_device = torch.log(torch.tensor(cfg.tau_init, dtype=torch.float32, device=device))
    else:
        init_log_tau_device = init_log_tau.detach().to(device).float()

    baseline_A = torch.zeros(C, r, dtype=torch.float32, device=device)

    # Global/no-personalization baseline on adapt_val
    baseline_metrics = evaluate_loader(
        frozen_model,
        adapt_val_loader,
        global_proto,
        proto_basis,
        baseline_A,
        init_log_tau_device,
        cfg,
        device,
    )
    baseline_score = personalization_selection_score(baseline_metrics)

    # Closed-form init from support set
    init_emp, present, _ = extract_confidence_weighted_prototypes(
        frozen_model,
        support_loader,
        global_proto,
        proto_basis,
        baseline_A,
        init_log_tau_device,
        num_classes=C,
        emb_dim=d,
        device=device,
        cfg=cfg,
        use_amp=cfg.amp,
        warmup=True,
    )
    init_A = solve_A_closed_form(
        init_emp.to(device),
        present.to(device),
        global_proto,
        proto_basis,
        cfg.lambda_A,
    ).detach()

    client_A = nn.Parameter(init_A.clone())
    log_tau = nn.Parameter(init_log_tau_device.clone())

    init_metrics = evaluate_loader(
        frozen_model,
        adapt_val_loader,
        global_proto,
        proto_basis,
        client_A.detach(),
        log_tau.detach(),
        cfg,
        device,
    )

    hps = choose_personalization_hparams(init_metrics["f1"], cfg)
    adapt_epochs = int(hps["epochs"])
    warm_epochs = int(hps["warm_epochs"])
    anchor_w = float(hps["anchor_w"])
    lr_A = float(hps["lr_A"])
    lr_tau = float(hps["lr_tau"])

    optimizer_A = torch.optim.AdamW(
        [{"params": [client_A], "lr": lr_A, "weight_decay": 1e-4}]
    )
    optimizer_joint = torch.optim.AdamW(
        [
            {"params": [client_A], "lr": lr_A, "weight_decay": 1e-4},
            {"params": [log_tau], "lr": lr_tau, "weight_decay": 0.0},
        ]
    )

    best = {
        "A": client_A.detach().cpu().clone(),
        "log_tau": log_tau.detach().cpu().clone(),
        "score": personalization_selection_score(init_metrics),
        "metrics": init_metrics,
    }

    no_improve = 0
    init_A_device = init_A.detach()

    for epoch in range(adapt_epochs):
        frozen_model.eval()
        optimizer = optimizer_A if epoch < warm_epochs else optimizer_joint

        for xb, yb in support_loader:
            xb = xb.to(device, non_blocking=True)
            yb = yb.to(device, non_blocking=True)

            optimizer.zero_grad(set_to_none=True)

            z = frozen_model(xb)
            P_i = personalized_prototypes_ccd(global_proto, proto_basis, client_A)
            logits = proto_logits(
                z,
                global_proto,
                proto_basis,
                client_A,
                log_tau,
                cfg,
            )

            loss_proto = F.cross_entropy(logits, yb, label_smoothing=0.02)
            align = 1.0 - torch.sum(z * P_i[yb], dim=1).mean()
            reg_A = client_A.pow(2).mean()
            anchor_A = (client_A - init_A_device).pow(2).mean()
            anchor_tau = (log_tau - init_log_tau_device).pow(2)

            loss = (
                loss_proto
                + cfg.lambda_align * align
                + cfg.lambda_A * reg_A
                + anchor_w * anchor_A
                + 0.5 * cfg.lambda_tau * anchor_tau
            )

            loss.backward()

            if cfg.grad_clip is not None and cfg.grad_clip > 0:
                if epoch < warm_epochs:
                    torch.nn.utils.clip_grad_norm_([client_A], cfg.grad_clip)
                else:
                    torch.nn.utils.clip_grad_norm_([client_A, log_tau], cfg.grad_clip)

            optimizer.step()

        val_metrics = evaluate_loader(
            frozen_model,
            adapt_val_loader,
            global_proto,
            proto_basis,
            client_A.detach(),
            log_tau.detach(),
            cfg,
            device,
        )

        score = personalization_selection_score(val_metrics)

        if score > best["score"]:
            best["score"] = score
            best["A"] = client_A.detach().cpu().clone()
            best["log_tau"] = log_tau.detach().cpu().clone()
            best["metrics"] = val_metrics
            no_improve = 0
        else:
            no_improve += 1

        if no_improve >= cfg.personalization_patience:
            break

    required_margin = cfg.personalization_gate_score_margin
    if baseline_metrics["f1"] >= cfg.easy_subject_f1_threshold:
        required_margin += cfg.personalization_gate_score_margin_easy

    use_personalization = (
        best["score"] > baseline_score + required_margin
        and best["metrics"]["f1"] >= baseline_metrics["f1"] + cfg.personalization_gate_f1_margin
    )

    if use_personalization:
        selected_A = best["A"]
        selected_log_tau = best["log_tau"]
        selected_metrics = best["metrics"]
    else:
        selected_A = baseline_A.detach().cpu().clone()
        selected_log_tau = init_log_tau_device.detach().cpu().clone()
        selected_metrics = baseline_metrics

    gate_info = {
        "used_personalization": bool(use_personalization),
        "baseline_metrics": baseline_metrics,
        "baseline_score": float(baseline_score),
        "personalized_metrics": best["metrics"],
        "personalized_score": float(best["score"]),
        "required_margin": float(required_margin),
    }

    return selected_A, selected_log_tau, selected_metrics, gate_info


## Fold training
One full LOSO fold training and evaluation pipeline.

In [ ]:
# ============================================================
# Fold training
# ============================================================

def train_one_fold(
    loaders: Dict[str, object],
    cfg: Config,
    device: torch.device,
) -> Dict[str, object]:
    num_classes = loaders["num_classes"]
    feature_dim = loaders["feature_dim"]
    heldout_id = loaders["heldout_id"]
    train_ids = loaders["train_ids"]
    train_clients = loaders["train_clients"]
    heldout = loaders["heldout"]

    global_model = TemporalEncoder(feature_dim, cfg).to(device)
    global_proto = F.normalize(torch.randn(num_classes, cfg.emb_dim, device=device), dim=1)
    proto_basis = orthonormal_random(cfg.emb_dim, cfg.proto_rank, device)

    client_A = {sid: torch.zeros(num_classes, cfg.proto_rank, dtype=torch.float32) for sid in train_ids}
    client_log_tau = {
        sid: torch.log(torch.tensor(cfg.tau_init, dtype=torch.float32))
        for sid in train_ids
    }

    best_snapshot = None
    best_score = -1.0
    history: List[Dict[str, float]] = []

    warmup_rounds = min(3, max(1, cfg.rounds // 4))
    round_bar = tqdm(range(1, cfg.rounds + 1), desc=f"Fold {heldout_id}", leave=True)

    for rnd in round_bar:
        local_state_dicts = []
        local_weights = []
        proto_summaries: Dict[str, Dict[str, torch.Tensor]] = {}

        for sid in train_ids:
            state_dict, A_local, log_tau_local, proto_summary = train_one_client(
                global_model=global_model,
                train_loader=train_clients[sid]["train"],
                val_loader=train_clients[sid]["val"],
                global_proto=global_proto,
                proto_basis=proto_basis,
                A_init=client_A[sid],
                log_tau_init=client_log_tau[sid],
                cfg=cfg,
                device=device,
                warmup_extract=(rnd <= warmup_rounds),
            )

            local_state_dicts.append(state_dict)
            local_weights.append(train_clients[sid]["n_train"])
            proto_summaries[sid] = proto_summary
            client_log_tau[sid] = log_tau_local.clone()

        avg_state = average_state_dicts(local_state_dicts, local_weights)
        global_model.load_state_dict(avg_state)

        global_proto, proto_basis, new_client_A = server_geometry_update(
            global_proto=global_proto,
            proto_basis=proto_basis,
            proto_summaries=proto_summaries,
            cfg=cfg,
            device=device,
        )
        client_A = new_client_A

        per_client_metrics: Dict[str, Dict[str, float]] = {}
        per_client_val_sizes: Dict[str, int] = {}

        with torch.no_grad():
            for sid in train_ids:
                metrics = evaluate_loader(
                    global_model,
                    train_clients[sid]["val"],
                    global_proto,
                    proto_basis,
                    client_A[sid].to(device),
                    client_log_tau[sid].to(device),
                    cfg,
                    device,
                )
                per_client_metrics[sid] = metrics
                per_client_val_sizes[sid] = len(train_clients[sid]["val"].dataset)

        mean_f1, mean_acc, mean_ece, score = aggregate_round_metrics(
            per_client_metrics, per_client_val_sizes, cfg
        )

        round_bar.set_postfix({
            "val_f1": f"{mean_f1:.4f}",
            "val_acc": f"{mean_acc:.4f}",
            "val_ece": f"{mean_ece:.4f}",
        })

        history.append({
            "round": rnd,
            "mean_val_f1": mean_f1,
            "mean_val_acc": mean_acc,
            "mean_val_ece": mean_ece,
        })

        if score > best_score:
            best_score = score
            best_snapshot = {
                "model_state": {k: v.detach().cpu().clone() for k, v in global_model.state_dict().items()},
                "global_proto": global_proto.detach().cpu().clone(),
                "proto_basis": proto_basis.detach().cpu().clone(),
                "client_A": {sid: a.detach().cpu().clone() for sid, a in client_A.items()},
                "client_log_tau": {sid: t.detach().cpu().clone() for sid, t in client_log_tau.items()},
                "history": copy.deepcopy(history),
            }

    assert best_snapshot is not None

    global_model.load_state_dict(best_snapshot["model_state"])
    global_proto = best_snapshot["global_proto"].to(device)
    proto_basis = best_snapshot["proto_basis"].to(device)
    saved_log_tau = best_snapshot["client_log_tau"]

    train_size_weights = torch.tensor(
        [train_clients[sid]["n_train"] for sid in train_ids],
        dtype=torch.float32,
        device=device,
    )
    tau_stack = torch.stack([saved_log_tau[sid].float().to(device) for sid in train_ids], dim=0)
    mean_log_tau = (train_size_weights * tau_stack).sum() / train_size_weights.sum()

    global_test = evaluate_loader(
        global_model,
        heldout["test"],
        global_proto,
        proto_basis,
        torch.zeros(num_classes, cfg.proto_rank, device=device),
        mean_log_tau,
        cfg,
        device,
    )

    best_A, best_log_tau, adapt_val_metrics, adapt_gate = adapt_heldout_client(
        model=global_model,
        support_loader=heldout["support"],
        adapt_val_loader=heldout["adapt_val"],
        global_proto=global_proto,
        proto_basis=proto_basis,
        cfg=cfg,
        device=device,
        init_log_tau=mean_log_tau.detach(),
    )

    personalized_test = evaluate_loader(
        global_model,
        heldout["test"],
        global_proto,
        proto_basis,
        best_A.to(device),
        best_log_tau.to(device),
        cfg,
        device,
    )

    return {
        "heldout_id": heldout_id,
        "summary": loaders["summary"],
        "history": best_snapshot["history"],
        "global_test": global_test,
        "personalized_test": personalized_test,
        "adapt_val": adapt_val_metrics,
        "adapt_gate": adapt_gate,
    }


## Experiment driver
Top-level helpers to launch the LOSO experiment and summarize results.

In [ ]:
# ============================================================
# Experiment driver
# ============================================================

def pretty_metric_line(name: str, metrics: Dict[str, float]) -> str:
    return (
        f"{name} -> ACC: {metrics['acc']:.4f}, "
        f"Macro-F1: {metrics['f1']:.4f}, "
        f"ECE: {metrics['ece']:.4f}, "
        f"Brier: {metrics['brier']:.4f}"
    )


def run_loso_experiment(cfg: Config) -> Dict[str, object]:
    set_seed(cfg.seed)
    device = torch.device(cfg.device)

    subject_files = discover_subject_files(cfg)
    subject_dfs = {sid: load_subject_dataframe(path, cfg) for sid, path in subject_files.items()}
    label_map = build_global_label_map(subject_dfs)
    subject_arrays = {sid: encode_subject(df, label_map) for sid, df in subject_dfs.items()}

    subject_ids = sorted(subject_arrays.keys())
    if cfg.run_single_heldout is not None:
        if cfg.run_single_heldout not in subject_ids:
            raise ValueError(f"Unknown held-out subject {cfg.run_single_heldout!r}. Available: {subject_ids}")
        subject_ids = [cfg.run_single_heldout]

    print("\n================ CONFIG ================")
    for k, v in asdict(cfg).items():
        print(f"{k}: {v}")

    print("\n================ SUBJECTS ================")
    print(subject_ids)
    print(f"Label map (activity_id -> class_idx): {label_map}")

    all_results = []
    for heldout_id in subject_ids:
        print(f"\n{'='*18} HELD-OUT {heldout_id} {'='*18}")
        fold = build_fold_data(subject_arrays, heldout_id, label_map, cfg)
        loaders = fold_to_loaders(fold, cfg)

        print("Fold summary:")
        for sid, stats in loaders["summary"].items():
            stats_str = ", ".join([f"{k}={v}" for k, v in stats.items()])
            print(f"  {sid}: {stats_str}")

        result = train_one_fold(loaders, cfg, device)
        all_results.append(result)

        print(pretty_metric_line("Global", result["global_test"]))
        print(pretty_metric_line("Personalized", result["personalized_test"]))

    global_metrics = {k: [] for k in ("acc", "f1", "ece", "brier")}
    pers_metrics = {k: [] for k in ("acc", "f1", "ece", "brier")}

    for res in all_results:
        for k in global_metrics.keys():
            global_metrics[k].append(res["global_test"][k])
            pers_metrics[k].append(res["personalized_test"][k])

    global_mean = {k: float(np.mean(v)) for k, v in global_metrics.items()}
    pers_mean = {k: float(np.mean(v)) for k, v in pers_metrics.items()}
    global_std = {k: float(np.std(v)) for k, v in global_metrics.items()}
    pers_std = {k: float(np.std(v)) for k, v in pers_metrics.items()}

    print("\n================ FINAL LOSO RESULTS ================")
    print(
        "Global      -> "
        f"ACC: {global_mean['acc']:.4f} ± {global_std['acc']:.4f}, "
        f"Macro-F1: {global_mean['f1']:.4f} ± {global_std['f1']:.4f}, "
        f"ECE: {global_mean['ece']:.4f} ± {global_std['ece']:.4f}, "
        f"Brier: {global_mean['brier']:.4f} ± {global_std['brier']:.4f}"
    )
    print(
        "Personalized-> "
        f"ACC: {pers_mean['acc']:.4f} ± {pers_std['acc']:.4f}, "
        f"Macro-F1: {pers_mean['f1']:.4f} ± {pers_std['f1']:.4f}, "
        f"ECE: {pers_mean['ece']:.4f} ± {pers_std['ece']:.4f}, "
        f"Brier: {pers_mean['brier']:.4f} ± {pers_std['brier']:.4f}"
    )

    gate = result["adapt_gate"]
    print(
        "Adapt gate -> "
        f"used_personalization={gate['used_personalization']} | "
        f"baseline_score={gate['baseline_score']:.4f} | "
        f"personalized_score={gate['personalized_score']:.4f} | "
        f"required_margin={gate['required_margin']:.4f}"
    )

    return {
        "folds": all_results,
        "global_mean": global_mean,
        "global_std": global_std,
        "personalized_mean": pers_mean,
        "personalized_std": pers_std,
        "label_map": label_map,
    }


## Minimal paper ablation harness 
Optional lightweight ablation harness for core paper variants.

In [ ]:
# ============================================================
# Minimal paper ablation harness (2-hour friendly)
# ============================================================

import json
import time
import hashlib
from collections import OrderedDict


@dataclass
class PaperAblationVariant:
    name: str
    description: str
    proto_rank: Optional[int] = None
    disable_deformation: bool = False
    fixed_basis: bool = False
    disable_support_adaptation: bool = False


ACTIVE_VARIANT = PaperAblationVariant(
    name="ccd_full",
    description="Full client-deformed prototype learning",
)


def set_active_variant(variant: PaperAblationVariant) -> None:
    global ACTIVE_VARIANT
    ACTIVE_VARIANT = variant


def get_active_variant() -> PaperAblationVariant:
    return ACTIVE_VARIANT


def stable_int_hash(text: str) -> int:
    return int(hashlib.md5(text.encode("utf-8")).hexdigest()[:8], 16)


def clone_cfg_for_variant(base_cfg: Config, variant: PaperAblationVariant, seed: int) -> Config:
    cfg = copy.deepcopy(base_cfg)
    cfg.seed = int(seed)
    if variant.proto_rank is not None:
        cfg.proto_rank = int(variant.proto_rank)
    return cfg


def init_basis_for_variant(emb_dim: int, proto_rank: int, device: torch.device, variant: PaperAblationVariant) -> torch.Tensor:
    if proto_rank <= 0:
        return torch.zeros(emb_dim, 0, device=device, dtype=torch.float32)
    if variant.fixed_basis:
        gen = torch.Generator(device=device)
        gen.manual_seed(stable_int_hash(variant.name) + 12345)
        rand = torch.randn(emb_dim, proto_rank, generator=gen, device=device)
        q, _ = torch.linalg.qr(rand)
        return q[:, :proto_rank].contiguous()
    return orthonormal_random(emb_dim, proto_rank, device)


def save_json(obj: object, path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, indent=2)


# ------------------------------------------------------------
# Variant-aware overrides
# ------------------------------------------------------------

def solve_A_closed_form(
    empirical_proto: torch.Tensor,
    present: torch.Tensor,
    global_proto: torch.Tensor,
    proto_basis: torch.Tensor,
    lambda_A: float,
) -> torch.Tensor:
    variant = get_active_variant()
    C = global_proto.shape[0]
    r = int(proto_basis.shape[1])

    if variant.disable_deformation or r == 0:
        return torch.zeros(C, r, device=global_proto.device, dtype=global_proto.dtype)

    BtB = proto_basis.T @ proto_basis
    inv = torch.linalg.inv(BtB + lambda_A * torch.eye(r, device=proto_basis.device, dtype=proto_basis.dtype))
    A = (empirical_proto - global_proto) @ proto_basis @ inv
    A = A * present.float().unsqueeze(1)
    return A


def train_one_client(
    global_model: nn.Module,
    train_loader: DataLoader,
    val_loader: DataLoader,
    global_proto: torch.Tensor,
    proto_basis: torch.Tensor,
    A_init: torch.Tensor,
    log_tau_init: torch.Tensor,
    cfg: Config,
    device: torch.device,
    warmup_extract: bool = False,
) -> Tuple[Dict[str, torch.Tensor], torch.Tensor, torch.Tensor, Dict[str, torch.Tensor]]:
    del val_loader
    variant = get_active_variant()

    local_model = clone_model(global_model).to(device)

    client_A = nn.Parameter(A_init.clone().to(device))
    log_tau = nn.Parameter(log_tau_init.clone().to(device))
    class_weight = make_class_weight_from_loader(train_loader, global_proto.shape[0], device, cfg)

    if variant.disable_deformation or cfg.proto_rank == 0:
        opt_params = list(local_model.parameters()) + [log_tau]
        train_A = False
    else:
        opt_params = list(local_model.parameters()) + [client_A, log_tau]
        train_A = True

    optimizer = torch.optim.AdamW(
        opt_params,
        lr=cfg.lr_encoder,
        weight_decay=cfg.weight_decay,
    )
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=max(1, cfg.local_epochs))
    scaler = torch.amp.GradScaler(enabled=(cfg.amp and device.type == "cuda"))

    autocast_device = "cuda" if device.type == "cuda" else "cpu"
    amp_enabled = cfg.amp and device.type == "cuda"

    optimizer_steps = 0
    scheduler_steps = 0

    for _epoch in range(cfg.local_epochs):
        local_model.train()

        for xb, yb in train_loader:
            xb = xb.to(device, non_blocking=True)
            yb = yb.to(device, non_blocking=True)

            optimizer.zero_grad(set_to_none=True)

            current_A = client_A if train_A else client_A.detach()

            with torch.amp.autocast(autocast_device, enabled=amp_enabled):
                z = local_model(xb)
                loss, _ = local_loss(
                    z, yb, global_proto, proto_basis, current_A, log_tau, cfg, class_weight=class_weight
                )

            if amp_enabled:
                scaler.scale(loss).backward()
                if cfg.grad_clip is not None and cfg.grad_clip > 0:
                    scaler.unscale_(optimizer)
                    grad_params = list(local_model.parameters()) + [log_tau]
                    if train_A:
                        grad_params.append(client_A)
                    torch.nn.utils.clip_grad_norm_(grad_params, cfg.grad_clip)
                old_scale = scaler.get_scale()
                scaler.step(optimizer)
                scaler.update()
                new_scale = scaler.get_scale()
                if new_scale >= old_scale:
                    optimizer_steps += 1
            else:
                loss.backward()
                if cfg.grad_clip is not None and cfg.grad_clip > 0:
                    grad_params = list(local_model.parameters()) + [log_tau]
                    if train_A:
                        grad_params.append(client_A)
                    torch.nn.utils.clip_grad_norm_(grad_params, cfg.grad_clip)
                optimizer.step()
                optimizer_steps += 1

        if optimizer_steps > scheduler_steps:
            scheduler.step()
            scheduler_steps += 1

    if not train_A:
        with torch.no_grad():
            client_A.zero_()

    emp, present, counts = extract_confidence_weighted_prototypes(
        model=local_model,
        loader=train_loader,
        global_proto=global_proto,
        proto_basis=proto_basis,
        client_A=client_A.detach(),
        log_tau=log_tau.detach(),
        num_classes=global_proto.shape[0],
        emb_dim=cfg.emb_dim,
        device=device,
        cfg=cfg,
        use_amp=cfg.amp,
        warmup=warmup_extract,
    )

    state_dict = {k: v.detach().cpu().clone() for k, v in local_model.state_dict().items()}
    del local_model

    return (
        state_dict,
        client_A.detach().cpu().clone(),
        log_tau.detach().cpu().clone(),
        {"empirical": emp, "present": present, "counts": counts},
    )


def server_geometry_update(
    global_proto: torch.Tensor,
    proto_basis: torch.Tensor,
    proto_summaries: Dict[str, Dict[str, torch.Tensor]],
    cfg: Config,
    device: torch.device,
) -> Tuple[torch.Tensor, torch.Tensor, Dict[str, torch.Tensor]]:
    variant = get_active_variant()

    old_P = global_proto.clone().to(device)
    old_B = proto_basis.clone().to(device)

    P = old_P.clone()
    B = old_B.clone()

    client_ids = list(proto_summaries.keys())
    H = torch.stack([proto_summaries[sid]["empirical"].to(device) for sid in client_ids], dim=0)
    M = torch.stack([proto_summaries[sid]["present"].to(device) for sid in client_ids], dim=0).bool()
    W = torch.stack([proto_summaries[sid]["counts"].to(device) for sid in client_ids], dim=0).float()
    W = torch.where(M, torch.sqrt(torch.clamp(W, min=0.0)), torch.zeros_like(W))

    K, C, d = H.shape
    r = B.shape[1]

    if variant.disable_deformation or r == 0:
        numer = torch.zeros_like(P)
        denom = torch.zeros(C, device=device)
        for i in range(K):
            numer += W[i].unsqueeze(1) * H[i]
            denom += W[i]
        keep = denom > 0
        if keep.any():
            P[keep] = numer[keep] / denom[keep].unsqueeze(1)
            P = F.normalize(P, dim=1)
        client_A = {sid: torch.zeros(C, r, dtype=torch.float32) for sid in client_ids}
        B = torch.zeros(d, r, device=device, dtype=P.dtype)
        return P.detach(), B.detach(), client_A

    A = torch.stack([solve_A_closed_form(H[i], M[i], P, B, cfg.lambda_A) for i in range(K)], dim=0)

    for _ in range(cfg.server_alt_iters):
        numer = torch.zeros_like(P)
        denom = torch.zeros(C, device=device)
        for i in range(K):
            numer += W[i].unsqueeze(1) * (H[i] - A[i] @ B.T)
            denom += W[i]

        keep = denom > 0
        if keep.any():
            P[keep] = numer[keep] / denom[keep].unsqueeze(1)
            P = F.normalize(P, dim=1)

        if not variant.fixed_basis:
            B_param = nn.Parameter(B.clone())
            opt = torch.optim.Adam([B_param], lr=cfg.server_geom_lr)
            for _step in range(cfg.server_geom_steps):
                opt.zero_grad(set_to_none=True)
                recon = P.unsqueeze(0) + torch.matmul(A, B_param.T)
                sq = ((H - recon) ** 2).sum(dim=2)
                data_term = (W * sq).sum() / (W.sum() + 1e-8)
                orth = ((B_param.T @ B_param) - torch.eye(r, device=device)).pow(2).mean()
                loss = data_term + cfg.lambda_B * orth
                loss.backward()
                opt.step()
            with torch.no_grad():
                q, _ = torch.linalg.qr(B_param.data)
                B = q[:, :r].contiguous()

        A = torch.stack([solve_A_closed_form(H[i], M[i], P, B, cfg.lambda_A) for i in range(K)], dim=0)

    P = F.normalize(cfg.server_proto_momentum * old_P + (1.0 - cfg.server_proto_momentum) * P, dim=1)
    if r > 0:
        if variant.fixed_basis:
            B = old_B
        else:
            B_blend = cfg.server_basis_momentum * old_B + (1.0 - cfg.server_basis_momentum) * B
            q, _ = torch.linalg.qr(B_blend)
            B = q[:, :r].contiguous()

    A = torch.stack([solve_A_closed_form(H[i], M[i], P, B, cfg.lambda_A) for i in range(K)], dim=0)
    client_A = {sid: A[idx].detach().cpu().clone() for idx, sid in enumerate(client_ids)}
    return P.detach(), B.detach(), client_A


def adapt_heldout_client(
    model: nn.Module,
    support_loader: DataLoader,
    adapt_val_loader: DataLoader,
    global_proto: torch.Tensor,
    proto_basis: torch.Tensor,
    cfg: Config,
    device: torch.device,
    init_log_tau: Optional[torch.Tensor] = None,
) -> Tuple[torch.Tensor, torch.Tensor, Dict[str, float], Dict[str, object]]:
    variant = get_active_variant()
    frozen_model = clone_model(model).to(device)
    frozen_model.eval()
    for p in frozen_model.parameters():
        p.requires_grad = False

    C, d = global_proto.shape
    r = proto_basis.shape[1]
    global_proto = global_proto.to(device)
    proto_basis = proto_basis.to(device)

    if init_log_tau is None:
        init_log_tau_device = torch.log(torch.tensor(cfg.tau_init, dtype=torch.float32, device=device))
    else:
        init_log_tau_device = init_log_tau.detach().to(device).float()

    baseline_A = torch.zeros(C, r, dtype=torch.float32, device=device)

    baseline_metrics = evaluate_loader(
        frozen_model, adapt_val_loader, global_proto, proto_basis, baseline_A, init_log_tau_device, cfg, device
    )
    baseline_score = personalization_selection_score(baseline_metrics)

    if variant.disable_support_adaptation:
        gate_info = {
            "used_personalization": False,
            "baseline_metrics": baseline_metrics,
            "baseline_score": float(baseline_score),
            "personalized_metrics": baseline_metrics,
            "personalized_score": float(baseline_score),
            "required_margin": 0.0,
        }
        return baseline_A.detach().cpu().clone(), init_log_tau_device.detach().cpu().clone(), baseline_metrics, gate_info

    init_emp, present, _ = extract_confidence_weighted_prototypes(
        frozen_model,
        support_loader,
        global_proto,
        proto_basis,
        baseline_A,
        init_log_tau_device,
        num_classes=C,
        emb_dim=d,
        device=device,
        cfg=cfg,
        use_amp=cfg.amp,
        warmup=True,
    )
    init_A = solve_A_closed_form(init_emp.to(device), present.to(device), global_proto, proto_basis, cfg.lambda_A).detach()

    client_A = nn.Parameter(init_A.clone())
    log_tau = nn.Parameter(init_log_tau_device.clone())

    init_metrics = evaluate_loader(
        frozen_model, adapt_val_loader, global_proto, proto_basis, client_A.detach(), log_tau.detach(), cfg, device
    )

    hps = choose_personalization_hparams(init_metrics["f1"], cfg)
    adapt_epochs = int(hps["epochs"])
    warm_epochs = int(hps["warm_epochs"])
    anchor_w = float(hps["anchor_w"])
    lr_A = float(hps["lr_A"])
    lr_tau = float(hps["lr_tau"])

    optimizer_A = torch.optim.AdamW([{"params": [client_A], "lr": lr_A, "weight_decay": 1e-4}])
    optimizer_joint = torch.optim.AdamW(
        [
            {"params": [client_A], "lr": lr_A, "weight_decay": 1e-4},
            {"params": [log_tau], "lr": lr_tau, "weight_decay": 0.0},
        ]
    )

    best = {
        "A": client_A.detach().cpu().clone(),
        "log_tau": log_tau.detach().cpu().clone(),
        "score": personalization_selection_score(init_metrics),
        "metrics": init_metrics,
    }

    no_improve = 0
    init_A_device = init_A.detach()

    for epoch in range(adapt_epochs):
        optimizer = optimizer_A if epoch < warm_epochs else optimizer_joint
        for xb, yb in support_loader:
            xb = xb.to(device, non_blocking=True)
            yb = yb.to(device, non_blocking=True)

            optimizer.zero_grad(set_to_none=True)
            z = frozen_model(xb)
            P_i = personalized_prototypes_ccd(global_proto, proto_basis, client_A)
            logits = proto_logits(z, global_proto, proto_basis, client_A, log_tau, cfg)

            loss_proto = F.cross_entropy(logits, yb, label_smoothing=0.02)
            align = 1.0 - torch.sum(z * P_i[yb], dim=1).mean()
            reg_A = client_A.pow(2).mean()
            anchor_A = (client_A - init_A_device).pow(2).mean()
            anchor_tau = (log_tau - init_log_tau_device).pow(2)

            loss = (
                loss_proto
                + cfg.lambda_align * align
                + cfg.lambda_A * reg_A
                + anchor_w * anchor_A
                + 0.5 * cfg.lambda_tau * anchor_tau
            )
            loss.backward()
            if cfg.grad_clip is not None and cfg.grad_clip > 0:
                if epoch < warm_epochs:
                    torch.nn.utils.clip_grad_norm_([client_A], cfg.grad_clip)
                else:
                    torch.nn.utils.clip_grad_norm_([client_A, log_tau], cfg.grad_clip)
            optimizer.step()

        val_metrics = evaluate_loader(
            frozen_model, adapt_val_loader, global_proto, proto_basis, client_A.detach(), log_tau.detach(), cfg, device
        )
        score = personalization_selection_score(val_metrics)
        if score > best["score"]:
            best["score"] = score
            best["A"] = client_A.detach().cpu().clone()
            best["log_tau"] = log_tau.detach().cpu().clone()
            best["metrics"] = val_metrics
            no_improve = 0
        else:
            no_improve += 1
        if no_improve >= cfg.personalization_patience:
            break

    required_margin = cfg.personalization_gate_score_margin
    if baseline_metrics["f1"] >= cfg.easy_subject_f1_threshold:
        required_margin += cfg.personalization_gate_score_margin_easy

    use_personalization = (
        best["score"] > baseline_score + required_margin
        and best["metrics"]["f1"] >= baseline_metrics["f1"] + cfg.personalization_gate_f1_margin
    )

    if use_personalization:
        selected_A = best["A"]
        selected_log_tau = best["log_tau"]
        selected_metrics = best["metrics"]
    else:
        selected_A = baseline_A.detach().cpu().clone()
        selected_log_tau = init_log_tau_device.detach().cpu().clone()
        selected_metrics = baseline_metrics

    gate_info = {
        "used_personalization": bool(use_personalization),
        "baseline_metrics": baseline_metrics,
        "baseline_score": float(baseline_score),
        "personalized_metrics": best["metrics"],
        "personalized_score": float(best["score"]),
        "required_margin": float(required_margin),
    }
    return selected_A, selected_log_tau, selected_metrics, gate_info


def train_one_fold(
    loaders: Dict[str, object],
    cfg: Config,
    device: torch.device,
) -> Dict[str, object]:
    variant = get_active_variant()
    num_classes = loaders["num_classes"]
    feature_dim = loaders["feature_dim"]
    heldout_id = loaders["heldout_id"]
    train_ids = loaders["train_ids"]
    train_clients = loaders["train_clients"]
    heldout = loaders["heldout"]

    global_model = TemporalEncoder(feature_dim, cfg).to(device)
    global_proto = F.normalize(torch.randn(num_classes, cfg.emb_dim, device=device), dim=1)
    proto_basis = init_basis_for_variant(cfg.emb_dim, cfg.proto_rank, device, variant)

    client_A = {sid: torch.zeros(num_classes, cfg.proto_rank, dtype=torch.float32) for sid in train_ids}
    client_log_tau = {sid: torch.log(torch.tensor(cfg.tau_init, dtype=torch.float32)) for sid in train_ids}

    best_snapshot = None
    best_score = -1.0

    warmup_rounds = min(2, max(1, cfg.rounds // 4))
    round_bar = tqdm(range(1, cfg.rounds + 1), desc=f"{variant.name} | {heldout_id}", leave=True)

    for rnd in round_bar:
        local_state_dicts = []
        local_weights = []
        proto_summaries: Dict[str, Dict[str, torch.Tensor]] = {}

        for sid in train_ids:
            state_dict, A_local, log_tau_local, proto_summary = train_one_client(
                global_model=global_model,
                train_loader=train_clients[sid]["train"],
                val_loader=train_clients[sid]["val"],
                global_proto=global_proto,
                proto_basis=proto_basis,
                A_init=client_A[sid],
                log_tau_init=client_log_tau[sid],
                cfg=cfg,
                device=device,
                warmup_extract=(rnd <= warmup_rounds),
            )

            local_state_dicts.append(state_dict)
            local_weights.append(train_clients[sid]["n_train"])
            proto_summaries[sid] = proto_summary
            client_log_tau[sid] = log_tau_local.clone()
            client_A[sid] = A_local.clone()

        avg_state = average_state_dicts(local_state_dicts, local_weights)
        global_model.load_state_dict(avg_state)

        global_proto, proto_basis, new_client_A = server_geometry_update(
            global_proto=global_proto,
            proto_basis=proto_basis,
            proto_summaries=proto_summaries,
            cfg=cfg,
            device=device,
        )
        client_A.update(new_client_A)

        per_client_metrics: Dict[str, Dict[str, float]] = {}
        per_client_val_sizes: Dict[str, int] = {}

        with torch.no_grad():
            for sid in train_ids:
                metrics = evaluate_loader(
                    global_model,
                    train_clients[sid]["val"],
                    global_proto,
                    proto_basis,
                    client_A[sid].to(device),
                    client_log_tau[sid].to(device),
                    cfg,
                    device,
                )
                per_client_metrics[sid] = metrics
                per_client_val_sizes[sid] = len(train_clients[sid]["val"].dataset)

        mean_f1, mean_acc, mean_ece, score = aggregate_round_metrics(per_client_metrics, per_client_val_sizes, cfg)

        round_bar.set_postfix({
            "val_f1": f"{mean_f1:.4f}",
            "val_acc": f"{mean_acc:.4f}",
            "val_ece": f"{mean_ece:.4f}",
        })

        if score > best_score:
            best_score = score
            best_snapshot = {
                "model_state": {k: v.detach().cpu().clone() for k, v in global_model.state_dict().items()},
                "global_proto": global_proto.detach().cpu().clone(),
                "proto_basis": proto_basis.detach().cpu().clone(),
                "client_A": {sid: a.detach().cpu().clone() for sid, a in client_A.items()},
                "client_log_tau": {sid: t.detach().cpu().clone() for sid, t in client_log_tau.items()},
            }

    assert best_snapshot is not None

    global_model.load_state_dict(best_snapshot["model_state"])
    global_proto = best_snapshot["global_proto"].to(device)
    proto_basis = best_snapshot["proto_basis"].to(device)
    saved_log_tau = best_snapshot["client_log_tau"]

    train_size_weights = torch.tensor([train_clients[sid]["n_train"] for sid in train_ids], dtype=torch.float32, device=device)
    tau_stack = torch.stack([saved_log_tau[sid].float().to(device) for sid in train_ids], dim=0)
    mean_log_tau = (train_size_weights * tau_stack).sum() / train_size_weights.sum()

    global_test = evaluate_loader(
        global_model, heldout["test"], global_proto, proto_basis,
        torch.zeros(num_classes, cfg.proto_rank, device=device), mean_log_tau, cfg, device
    )

    best_A, best_log_tau, adapt_val_metrics, adapt_gate = adapt_heldout_client(
        model=global_model,
        support_loader=heldout["support"],
        adapt_val_loader=heldout["adapt_val"],
        global_proto=global_proto,
        proto_basis=proto_basis,
        cfg=cfg,
        device=device,
        init_log_tau=mean_log_tau.detach(),
    )

    personalized_test = evaluate_loader(
        global_model, heldout["test"], global_proto, proto_basis, best_A.to(device), best_log_tau.to(device), cfg, device
    )

    return {
        "heldout_id": heldout_id,
        "summary": loaders["summary"],
        "global_test": global_test,
        "personalized_test": personalized_test,
        "adapt_val": adapt_val_metrics,
        "adapt_gate": adapt_gate,
    }


# ------------------------------------------------------------
# Minimal paper suite
# ------------------------------------------------------------

def build_minimal_paper_suite() -> "OrderedDict[str, List[PaperAblationVariant]]":
    suites = OrderedDict()
    suites["paper_essential"] = [
        PaperAblationVariant("ccd_full", "Full method"),
        PaperAblationVariant("no_deformation", "A_i = 0, no deformation", proto_rank=0, disable_deformation=True),
        PaperAblationVariant("fixed_basis", "Fixed shared basis, no basis learning", fixed_basis=True),
        PaperAblationVariant("no_support_adaptation", "No held-out support adaptation", disable_support_adaptation=True),
    ]
    return suites


def run_loso_experiment_variant(cfg: Config, variant: PaperAblationVariant) -> Dict[str, object]:
    set_seed(cfg.seed)
    set_active_variant(variant)
    device = torch.device(cfg.device)

    subject_files = discover_subject_files(cfg)
    subject_dfs = {sid: load_subject_dataframe(path, cfg) for sid, path in subject_files.items()}
    label_map = build_global_label_map(subject_dfs)
    subject_arrays = {sid: encode_subject(df, label_map) for sid, df in subject_dfs.items()}

    subject_ids = sorted(subject_arrays.keys())
    if cfg.run_single_heldout is not None:
        subject_ids = [cfg.run_single_heldout]

    all_results = []
    for heldout_id in subject_ids:
        print(f"\n{'='*18} {variant.name} | HELD-OUT {heldout_id} {'='*18}")
        fold = build_fold_data(subject_arrays, heldout_id, label_map, cfg)
        loaders = fold_to_loaders(fold, cfg)
        result = train_one_fold(loaders, cfg, device)
        all_results.append(result)
        print(pretty_metric_line("Global", result["global_test"]))
        print(pretty_metric_line("Personalized", result["personalized_test"]))

    global_metrics = {k: [] for k in ("acc", "f1", "ece", "brier")}
    pers_metrics = {k: [] for k in ("acc", "f1", "ece", "brier")}
    for res in all_results:
        for k in global_metrics.keys():
            global_metrics[k].append(res["global_test"][k])
            pers_metrics[k].append(res["personalized_test"][k])

    global_mean = {k: float(np.mean(v)) for k, v in global_metrics.items()}
    pers_mean = {k: float(np.mean(v)) for k, v in pers_metrics.items()}
    global_std = {k: float(np.std(v)) for k, v in global_metrics.items()}
    pers_std = {k: float(np.std(v)) for k, v in pers_metrics.items()}

    return {
        "variant": asdict(variant),
        "cfg_snapshot": asdict(cfg),
        "folds": all_results,
        "global_mean": global_mean,
        "global_std": global_std,
        "personalized_mean": pers_mean,
        "personalized_std": pers_std,
        "label_map": label_map,
    }


def run_minimal_paper_ablation(base_cfg: Config, seeds: List[int], save_root: str) -> Dict[str, object]:
    save_root = Path(save_root)
    save_root.mkdir(parents=True, exist_ok=True)

    suite = build_minimal_paper_suite()["paper_essential"]
    all_rows = []

    for variant in suite:
        for seed in seeds:
            cfg = clone_cfg_for_variant(base_cfg, variant, seed)
            print(f"\n######## variant={variant.name} | seed={seed} ########")
            out = run_loso_experiment_variant(cfg, variant)

            variant_dir = save_root / variant.name / f"seed_{seed}"
            variant_dir.mkdir(parents=True, exist_ok=True)
            save_json(out, variant_dir / "results.json")

            for fold in out["folds"]:
                all_rows.append({
                    "variant": variant.name,
                    "description": variant.description,
                    "seed": seed,
                    "heldout_id": fold["heldout_id"],
                    "global_acc": fold["global_test"]["acc"],
                    "global_f1": fold["global_test"]["f1"],
                    "global_ece": fold["global_test"]["ece"],
                    "personalized_acc": fold["personalized_test"]["acc"],
                    "personalized_f1": fold["personalized_test"]["f1"],
                    "personalized_ece": fold["personalized_test"]["ece"],
                    "used_personalization": int(fold["adapt_gate"]["used_personalization"]),
                    "proto_rank": cfg.proto_rank,
                    "disable_deformation": int(variant.disable_deformation),
                    "fixed_basis": int(variant.fixed_basis),
                    "disable_support_adaptation": int(variant.disable_support_adaptation),
                })

    fold_df = pd.DataFrame(all_rows)
    fold_df.to_csv(save_root / "paper_essential_fold_results.csv", index=False)

    summary_df = fold_df.groupby(["variant", "description"], as_index=False).agg({
        "global_acc": ["mean", "std"],
        "global_f1": ["mean", "std"],
        "global_ece": ["mean", "std"],
        "personalized_acc": ["mean", "std"],
        "personalized_f1": ["mean", "std"],
        "personalized_ece": ["mean", "std"],
        "used_personalization": "mean",
        "proto_rank": "first",
    })
    summary_df.columns = [
        "_".join([x for x in c if x]).strip("_") if isinstance(c, tuple) else c
        for c in summary_df.columns.to_flat_index()
    ]
    summary_df.to_csv(save_root / "paper_essential_summary.csv", index=False)

    print("\nSaved:", save_root / "paper_essential_fold_results.csv")
    print("Saved:", save_root / "paper_essential_summary.csv")
    return {"fold_df": fold_df, "summary_df": summary_df}


## Minimal paper ablation run cell
This is the reduced paper ablation version.

Default mode is intentionally constrained so it can fit roughly within ~2 hours
on a setup where your original full LOSO run takes ~1.5 hours:
  - one representative held-out subject only
  - one seed
  - four essential ablations only
  - lighter training budget

Essential ablations included:
  1) ccd_full
  2) no_deformation
  3) fixed_basis
  4) no_support_adaptation

For the final paper after the pilot works:
  - set run_single_heldout=None
  - increase SEEDS to [42, 52, 62]
  - optionally increase rounds/local_epochs back upward

In [ ]:
base_cfg = Config(
    data_dir="/content/drive/MyDrive/PAMAP2_Dataset/Protocol",
    save_dir="/content/drive/MyDrive/ablation",
    run_single_heldout=None,

    rounds=8,
    local_epochs=4,
    personalization_epochs=6,
    personalization_epochs_min=2,
    personalization_epochs_max=6,
    personalization_patience=2,

    server_geom_steps=8,
    server_alt_iters=2,

    lr_personal=7.5e-3,
)

SEEDS = [42]

run_minimal_paper_ablation(
    base_cfg=base_cfg,
    seeds=SEEDS,
    save_root=str(Path(base_cfg.save_dir) / "paper_essential"),
)


## Rank and support sweep utilities
Optional sweep utilities for prototype rank and held-out support ratio. The example runs are preserved as commented templates to avoid accidental long executions.

In [ ]:
# ============================================================
# Rank sweep + Support sweep (run AFTER your existing notebook)
# ============================================================

import matplotlib.pyplot as plt
from pathlib import Path
import copy
import json

# ------------------------------------------------------------
# Small helpers
# ------------------------------------------------------------

def _safe_save_json(obj, path: Path):
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, indent=2)

def _flatten_metric_rows(exp_name: str, setting_name: str, setting_value, seed: int, out: Dict[str, object]):
    rows = []
    cfg_snapshot = out["cfg_snapshot"]
    variant_name = out["variant"]["name"]
    variant_desc = out["variant"]["description"]

    for fold in out["folds"]:
        rows.append({
            "experiment": exp_name,
            "setting_name": setting_name,
            "setting_value": setting_value,
            "variant": variant_name,
            "description": variant_desc,
            "seed": seed,
            "heldout_id": fold["heldout_id"],

            "global_acc": fold["global_test"]["acc"],
            "global_f1": fold["global_test"]["f1"],
            "global_ece": fold["global_test"]["ece"],
            "global_brier": fold["global_test"]["brier"],

            "personalized_acc": fold["personalized_test"]["acc"],
            "personalized_f1": fold["personalized_test"]["f1"],
            "personalized_ece": fold["personalized_test"]["ece"],
            "personalized_brier": fold["personalized_test"]["brier"],

            "adapt_val_acc": fold["adapt_val"]["acc"],
            "adapt_val_f1": fold["adapt_val"]["f1"],
            "adapt_val_ece": fold["adapt_val"]["ece"],
            "adapt_val_brier": fold["adapt_val"]["brier"],

            "used_personalization": int(fold["adapt_gate"]["used_personalization"]),
            "proto_rank": cfg_snapshot["proto_rank"],
            "heldout_support_ratio": cfg_snapshot["heldout_support_ratio"],
            "heldout_adapt_val_ratio": cfg_snapshot["heldout_adapt_val_ratio"],
            "heldout_test_ratio": cfg_snapshot["heldout_test_ratio"],
        })
    return rows


def _summarize_sweep_df(df: pd.DataFrame, group_cols: List[str]) -> pd.DataFrame:
    metric_cols = [
        "global_acc", "global_f1", "global_ece", "global_brier",
        "personalized_acc", "personalized_f1", "personalized_ece", "personalized_brier",
        "adapt_val_acc", "adapt_val_f1", "adapt_val_ece", "adapt_val_brier",
        "used_personalization",
    ]

    summary = df.groupby(group_cols, as_index=False).agg({
        **{c: ["mean", "std"] for c in metric_cols},
        "proto_rank": "first",
        "heldout_support_ratio": "first",
        "heldout_adapt_val_ratio": "first",
        "heldout_test_ratio": "first",
    })

    summary.columns = [
        "_".join([x for x in col if x]).strip("_") if isinstance(col, tuple) else col
        for col in summary.columns.to_flat_index()
    ]
    return summary


def _plot_sweep(summary_df: pd.DataFrame, x_col: str, out_dir: Path, prefix: str):
    out_dir.mkdir(parents=True, exist_ok=True)

    # Personalized F1
    plt.figure(figsize=(7, 5))
    x = summary_df[x_col].values
    y = summary_df["personalized_f1_mean"].values
    yerr = summary_df["personalized_f1_std"].fillna(0.0).values
    plt.plot(x, y, marker="o")
    plt.fill_between(x, y - yerr, y + yerr, alpha=0.2)
    plt.xlabel(x_col)
    plt.ylabel("Personalized Macro-F1")
    plt.title(f"{prefix}: Personalized Macro-F1")
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(out_dir / f"{prefix.lower().replace(' ', '_')}_f1.png", dpi=220)
    plt.close()

    # Personalized ECE
    plt.figure(figsize=(7, 5))
    y = summary_df["personalized_ece_mean"].values
    yerr = summary_df["personalized_ece_std"].fillna(0.0).values
    plt.plot(x, y, marker="o")
    plt.fill_between(x, y - yerr, y + yerr, alpha=0.2)
    plt.xlabel(x_col)
    plt.ylabel("Personalized ECE")
    plt.title(f"{prefix}: Personalized ECE")
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(out_dir / f"{prefix.lower().replace(' ', '_')}_ece.png", dpi=220)
    plt.close()


# ------------------------------------------------------------
# Rank sweep
# ------------------------------------------------------------

def build_rank_sweep_variants(rank_values: List[int]) -> List[PaperAblationVariant]:
    variants = []
    for r in rank_values:
        if r == 0:
            variants.append(
                PaperAblationVariant(
                    name=f"rank_{r}",
                    description=f"Rank {r} (no deformation)",
                    proto_rank=0,
                    disable_deformation=True,
                )
            )
        else:
            variants.append(
                PaperAblationVariant(
                    name=f"rank_{r}",
                    description=f"Rank {r}",
                    proto_rank=int(r),
                    disable_deformation=False,
                )
            )
    return variants


def run_rank_sweep(
    base_cfg: Config,
    rank_values: List[int],
    seeds: List[int],
    save_root: str,
) -> Dict[str, pd.DataFrame]:
    save_root = Path(save_root)
    save_root.mkdir(parents=True, exist_ok=True)

    all_rows = []
    variants = build_rank_sweep_variants(rank_values)

    for variant in variants:
        for seed in seeds:
            cfg = clone_cfg_for_variant(base_cfg, variant, seed)

            print(f"\n######## RANK SWEEP | rank={cfg.proto_rank} | seed={seed} ########")
            out = run_loso_experiment_variant(cfg, variant)

            rank_dir = save_root / variant.name / f"seed_{seed}"
            rank_dir.mkdir(parents=True, exist_ok=True)
            _safe_save_json(out, rank_dir / "results.json")

            setting_value = 0 if variant.disable_deformation else cfg.proto_rank
            rows = _flatten_metric_rows(
                exp_name="rank_sweep",
                setting_name="rank",
                setting_value=setting_value,
                seed=seed,
                out=out,
            )
            all_rows.extend(rows)

    fold_df = pd.DataFrame(all_rows)
    fold_csv = save_root / "rank_sweep_fold_results.csv"
    fold_df.to_csv(fold_csv, index=False)

    summary_df = _summarize_sweep_df(
        fold_df,
        group_cols=["experiment", "setting_name", "setting_value"]
    ).sort_values("setting_value")
    summary_csv = save_root / "rank_sweep_summary.csv"
    summary_df.to_csv(summary_csv, index=False)

    _plot_sweep(summary_df, x_col="setting_value", out_dir=save_root, prefix="Rank Sweep")

    print("\nSaved:", fold_csv)
    print("Saved:", summary_csv)
    print("Saved plots in:", save_root)

    return {
        "fold_df": fold_df,
        "summary_df": summary_df,
    }


# ------------------------------------------------------------
# Support sweep
# ------------------------------------------------------------

def make_support_cfg(base_cfg: Config, support_ratio: float, seed: int) -> Config:
    cfg = copy.deepcopy(base_cfg)
    cfg.seed = int(seed)

    adapt_ratio = float(base_cfg.heldout_adapt_val_ratio)
    test_ratio = 1.0 - support_ratio - adapt_ratio
    if test_ratio <= 0:
        raise ValueError(
            f"Invalid support ratio {support_ratio:.3f}; "
            f"support + adapt must be < 1.0. Current adapt={adapt_ratio:.3f}"
        )

    cfg.heldout_support_ratio = float(support_ratio)
    cfg.heldout_adapt_val_ratio = float(adapt_ratio)
    cfg.heldout_test_ratio = float(test_ratio)
    return cfg


def run_support_sweep(
    base_cfg: Config,
    support_values: List[float],
    seeds: List[int],
    save_root: str,
) -> Dict[str, pd.DataFrame]:
    save_root = Path(save_root)
    save_root.mkdir(parents=True, exist_ok=True)

    all_rows = []
    full_variant = PaperAblationVariant(
        name="ccd_full",
        description="Full client-deformed prototype learning",
    )

    for support_ratio in support_values:
        for seed in seeds:
            cfg = make_support_cfg(base_cfg, support_ratio=support_ratio, seed=seed)

            print(f"\n######## SUPPORT SWEEP | support={support_ratio:.2f} | seed={seed} ########")
            out = run_loso_experiment_variant(cfg, full_variant)

            sup_tag = f"support_{int(round(100 * support_ratio)):02d}"
            sup_dir = save_root / sup_tag / f"seed_{seed}"
            sup_dir.mkdir(parents=True, exist_ok=True)
            _safe_save_json(out, sup_dir / "results.json")

            rows = _flatten_metric_rows(
                exp_name="support_sweep",
                setting_name="support_ratio",
                setting_value=float(support_ratio),
                seed=seed,
                out=out,
            )
            all_rows.extend(rows)

    fold_df = pd.DataFrame(all_rows)
    fold_csv = save_root / "support_sweep_fold_results.csv"
    fold_df.to_csv(fold_csv, index=False)

    summary_df = _summarize_sweep_df(
        fold_df,
        group_cols=["experiment", "setting_name", "setting_value"]
    ).sort_values("setting_value")
    summary_csv = save_root / "support_sweep_summary.csv"
    summary_df.to_csv(summary_csv, index=False)

    _plot_sweep(summary_df, x_col="setting_value", out_dir=save_root, prefix="Support Sweep")

    print("\nSaved:", fold_csv)
    print("Saved:", summary_csv)
    print("Saved plots in:", save_root)

    return {
        "fold_df": fold_df,
        "summary_df": summary_df,
    }


# ------------------------------------------------------------
# Example runs
# ------------------------------------------------------------
# Use these after defining base_cfg exactly the way you want.

# Example 1: rank sweep
# rank_results = run_rank_sweep(
#     base_cfg=base_cfg,
#     rank_values=[0, 2, 4, 8, 16],
#     seeds=[42],
#     save_root=str(Path(base_cfg.save_dir) / "rank_sweep"),
# )

# Example 2: support sweep
# support_results = run_support_sweep(
#     base_cfg=base_cfg,
#     support_values=[0.05, 0.10, 0.20, 0.30],
#     seeds=[42],
#     save_root=str(Path(base_cfg.save_dir) / "support_sweep"),
# )

# Example 3: run both
# rank_results = run_rank_sweep(
#     base_cfg=base_cfg,
#     rank_values=[0, 2, 4, 8, 16],
#     seeds=[42],
#     save_root=str(Path(base_cfg.save_dir) / "rank_sweep"),
# )
#
# support_results = run_support_sweep(
#     base_cfg=base_cfg,
#     support_values=[0.05, 0.10, 0.20, 0.30],
#     seeds=[42],
#     save_root=str(Path(base_cfg.save_dir) / "support_sweep"),
# )

# rank_results = run_rank_sweep(
#     base_cfg=base_cfg,
#     rank_values=[0, 2, 4, 8, 16],
#     seeds=[42],
#     save_root=str(Path(base_cfg.save_dir) / "rank_sweep"),
# )


## Missing sensors / channels ablation
Optional ablation utilities for held-out missing-sensor robustness.

In [ ]:
# ============================================================
# Missing sensors / channels ablation
# Run this AFTER your current notebook code
# ============================================================

import copy
import json
from pathlib import Path
from dataclasses import dataclass, asdict

# ------------------------------------------------------------
# Variant
# ------------------------------------------------------------

@dataclass
class MissingSensorVariant:
    name: str
    description: str
    missing_sensor_rate: float = 0.0  # fraction of feature channels to zero on held-out subject


ACTIVE_MISSING_SENSOR_VARIANT = MissingSensorVariant(
    name="missing_00",
    description="No missing sensors",
    missing_sensor_rate=0.0,
)


def set_active_missing_sensor_variant(variant: MissingSensorVariant) -> None:
    global ACTIVE_MISSING_SENSOR_VARIANT
    ACTIVE_MISSING_SENSOR_VARIANT = variant


def get_active_missing_sensor_variant() -> MissingSensorVariant:
    return ACTIVE_MISSING_SENSOR_VARIANT


# ------------------------------------------------------------
# Helpers
# ------------------------------------------------------------

def _ms_safe_save_json(obj, path: Path):
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, indent=2)


def _apply_missing_mask_to_windows(windows: np.ndarray, mask_dims: np.ndarray) -> np.ndarray:
    if len(windows) == 0 or len(mask_dims) == 0:
        return windows.copy()
    out = windows.copy()
    out[..., mask_dims] = 0.0
    return out


def _apply_missing_sensor_to_fold(
    fold: FoldData,
    missing_sensor_rate: float,
    seed: int,
) -> FoldData:
    """
    Zero out the same subset of held-out feature channels in:
      - support
      - adapt_val
      - test
    Training clients are untouched.
    """
    if missing_sensor_rate <= 0.0:
        return fold

    fold = copy.deepcopy(fold)
    feat_dim = fold.feature_dim
    n_drop = max(1, int(round(feat_dim * missing_sensor_rate)))

    rng = np.random.RandomState(seed + stable_int_hash(f"{fold.heldout_id}|missing|{missing_sensor_rate}"))
    mask_dims = np.sort(rng.choice(np.arange(feat_dim), size=n_drop, replace=False))

    for split_name in ("support", "adapt_val", "test"):
        part = fold.heldout_subject[split_name]
        fold.heldout_subject[split_name] = SubjectPartition(
            windows=_apply_missing_mask_to_windows(part.windows, mask_dims),
            labels=part.labels.copy(),
        )

    print(f"[MISSING SENSOR] heldout={fold.heldout_id} | dropped {n_drop}/{feat_dim} channels | idx={mask_dims.tolist()}")
    return fold


def _ms_flatten_rows(exp_name: str, setting_value: float, seed: int, out: Dict[str, object]):
    rows = []
    for fold in out["folds"]:
        rows.append({
            "experiment": exp_name,
            "missing_sensor_rate": setting_value,
            "variant": out["variant"]["name"],
            "description": out["variant"]["description"],
            "seed": seed,
            "heldout_id": fold["heldout_id"],

            "global_acc": fold["global_test"]["acc"],
            "global_f1": fold["global_test"]["f1"],
            "global_ece": fold["global_test"]["ece"],
            "global_brier": fold["global_test"]["brier"],

            "personalized_acc": fold["personalized_test"]["acc"],
            "personalized_f1": fold["personalized_test"]["f1"],
            "personalized_ece": fold["personalized_test"]["ece"],
            "personalized_brier": fold["personalized_test"]["brier"],

            "adapt_val_acc": fold["adapt_val"]["acc"],
            "adapt_val_f1": fold["adapt_val"]["f1"],
            "adapt_val_ece": fold["adapt_val"]["ece"],
            "adapt_val_brier": fold["adapt_val"]["brier"],

            "used_personalization": int(fold["adapt_gate"]["used_personalization"]),
        })
    return rows


def _ms_summarize(df: pd.DataFrame) -> pd.DataFrame:
    summary = df.groupby(["experiment", "missing_sensor_rate"], as_index=False).agg({
        "global_acc": ["mean", "std"],
        "global_f1": ["mean", "std"],
        "global_ece": ["mean", "std"],
        "global_brier": ["mean", "std"],
        "personalized_acc": ["mean", "std"],
        "personalized_f1": ["mean", "std"],
        "personalized_ece": ["mean", "std"],
        "personalized_brier": ["mean", "std"],
        "adapt_val_acc": ["mean", "std"],
        "adapt_val_f1": ["mean", "std"],
        "adapt_val_ece": ["mean", "std"],
        "adapt_val_brier": ["mean", "std"],
        "used_personalization": "mean",
    })

    summary.columns = [
        "_".join([x for x in c if x]).strip("_") if isinstance(c, tuple) else c
        for c in summary.columns.to_flat_index()
    ]
    return summary.sort_values("missing_sensor_rate")


def _ms_plot(summary_df: pd.DataFrame, out_dir: Path):
    out_dir.mkdir(parents=True, exist_ok=True)

    x = summary_df["missing_sensor_rate"].values

    # Personalized F1
    plt.figure(figsize=(7, 5))
    y = summary_df["personalized_f1_mean"].values
    yerr = summary_df["personalized_f1_std"].fillna(0.0).values
    plt.plot(x, y, marker="o")
    plt.fill_between(x, y - yerr, y + yerr, alpha=0.2)
    plt.xlabel("Missing sensor rate")
    plt.ylabel("Personalized Macro-F1")
    plt.title("Missing Sensors / Channels: Personalized Macro-F1")
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(out_dir / "missing_sensor_f1.png", dpi=220)
    plt.close()

    # Personalized ECE
    plt.figure(figsize=(7, 5))
    y = summary_df["personalized_ece_mean"].values
    yerr = summary_df["personalized_ece_std"].fillna(0.0).values
    plt.plot(x, y, marker="o")
    plt.fill_between(x, y - yerr, y + yerr, alpha=0.2)
    plt.xlabel("Missing sensor rate")
    plt.ylabel("Personalized ECE")
    plt.title("Missing Sensors / Channels: Personalized ECE")
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(out_dir / "missing_sensor_ece.png", dpi=220)
    plt.close()


# ------------------------------------------------------------
# Runner
# ------------------------------------------------------------

def run_loso_experiment_missing_sensor_variant(
    cfg: Config,
    variant: MissingSensorVariant,
) -> Dict[str, object]:
    set_seed(cfg.seed)
    set_active_missing_sensor_variant(variant)
    device = torch.device(cfg.device)

    subject_files = discover_subject_files(cfg)
    subject_dfs = {sid: load_subject_dataframe(path, cfg) for sid, path in subject_files.items()}
    label_map = build_global_label_map(subject_dfs)
    subject_arrays = {sid: encode_subject(df, label_map) for sid, df in subject_dfs.items()}

    subject_ids = sorted(subject_arrays.keys())
    if cfg.run_single_heldout is not None:
        if cfg.run_single_heldout not in subject_ids:
            raise ValueError(f"Unknown held-out subject {cfg.run_single_heldout!r}. Available: {subject_ids}")
        subject_ids = [cfg.run_single_heldout]

    all_results = []
    for heldout_id in subject_ids:
        print(f"\n{'='*18} MISSING SENSOR={variant.missing_sensor_rate:.2f} | HELD-OUT {heldout_id} {'='*18}")
        fold = build_fold_data(subject_arrays, heldout_id, label_map, cfg)
        fold = _apply_missing_sensor_to_fold(
            fold=fold,
            missing_sensor_rate=variant.missing_sensor_rate,
            seed=cfg.seed,
        )
        loaders = fold_to_loaders(fold, cfg)
        result = train_one_fold(loaders, cfg, device)
        all_results.append(result)

        print(pretty_metric_line("Global", result["global_test"]))
        print(pretty_metric_line("Personalized", result["personalized_test"]))

    global_metrics = {k: [] for k in ("acc", "f1", "ece", "brier")}
    pers_metrics = {k: [] for k in ("acc", "f1", "ece", "brier")}

    for res in all_results:
        for k in global_metrics.keys():
            global_metrics[k].append(res["global_test"][k])
            pers_metrics[k].append(res["personalized_test"][k])

    global_mean = {k: float(np.mean(v)) for k, v in global_metrics.items()}
    pers_mean = {k: float(np.mean(v)) for k, v in pers_metrics.items()}
    global_std = {k: float(np.std(v)) for k, v in global_metrics.items()}
    pers_std = {k: float(np.std(v)) for k, v in pers_metrics.items()}

    return {
        "variant": asdict(variant),
        "cfg_snapshot": asdict(cfg),
        "folds": all_results,
        "global_mean": global_mean,
        "global_std": global_std,
        "personalized_mean": pers_mean,
        "personalized_std": pers_std,
        "label_map": label_map,
    }


def run_missing_sensor_ablation(
    base_cfg: Config,
    missing_rates: List[float],
    seeds: List[int],
    save_root: str,
) -> Dict[str, pd.DataFrame]:
    """
    Example missing_rates:
      [0.0, 0.10, 0.20, 0.30]
    """
    save_root = Path(save_root)
    save_root.mkdir(parents=True, exist_ok=True)

    all_rows = []

    for missing_rate in missing_rates:
        variant = MissingSensorVariant(
            name=f"missing_{int(round(100 * missing_rate)):02d}",
            description=f"{int(round(100 * missing_rate))}% held-out channels missing",
            missing_sensor_rate=float(missing_rate),
        )

        for seed in seeds:
            cfg = copy.deepcopy(base_cfg)
            cfg.seed = int(seed)

            print(f"\n######## MISSING SENSOR | rate={missing_rate:.2f} | seed={seed} ########")
            out = run_loso_experiment_missing_sensor_variant(cfg, variant)

            run_dir = save_root / variant.name / f"seed_{seed}"
            run_dir.mkdir(parents=True, exist_ok=True)
            _ms_safe_save_json(out, run_dir / "results.json")

            all_rows.extend(
                _ms_flatten_rows(
                    exp_name="missing_sensor_ablation",
                    setting_value=missing_rate,
                    seed=seed,
                    out=out,
                )
            )

    fold_df = pd.DataFrame(all_rows)
    fold_csv = save_root / "missing_sensor_fold_results.csv"
    fold_df.to_csv(fold_csv, index=False)

    summary_df = _ms_summarize(fold_df)
    summary_csv = save_root / "missing_sensor_summary.csv"
    summary_df.to_csv(summary_csv, index=False)

    _ms_plot(summary_df, save_root)

    print("\nSaved:", fold_csv)
    print("Saved:", summary_csv)
    print("Saved plots in:", save_root)

    return {
        "fold_df": fold_df,
        "summary_df": summary_df,
    }


# ------------------------------------------------------------
# Example run
# ------------------------------------------------------------

# missing_sensor_results = run_missing_sensor_ablation(
#     base_cfg=base_cfg,
#     missing_rates=[0.0, 0.10, 0.20, 0.30],
#     seeds=[42],
#     save_root=str(Path(base_cfg.save_dir) / "missing_sensor_ablation"),
# )


## Resume-safe missing sensor ablation
Optional resume-safe missing-sensor ablation helpers. Example run cells are preserved as commented templates.

In [ ]:
# ============================================================
# Resume-safe missing sensor ablation
# - skips completed runs if results.json already exists
# - rebuilds fold CSV / summary CSV / plots from all saved runs
# ============================================================

import copy
import json
from pathlib import Path

def _ms_load_json(path: Path):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)

def _ms_collect_saved_rows(save_root: Path):
    rows = []
    for results_path in sorted(save_root.glob("missing_*/seed_*/results.json")):
        try:
            out = _ms_load_json(results_path)
            rate = float(out["variant"]["missing_sensor_rate"])
            seed = int(out["cfg_snapshot"]["seed"])
            rows.extend(
                _ms_flatten_rows(
                    exp_name="missing_sensor_ablation",
                    setting_value=rate,
                    seed=seed,
                    out=out,
                )
            )
        except Exception as exc:
            print(f"[WARN] Could not read {results_path}: {exc}")
    return rows

def rebuild_missing_sensor_outputs(save_root: str):
    save_root = Path(save_root)
    save_root.mkdir(parents=True, exist_ok=True)

    all_rows = _ms_collect_saved_rows(save_root)
    if len(all_rows) == 0:
        print("No saved missing-sensor results found.")
        return None

    fold_df = pd.DataFrame(all_rows)
    fold_csv = save_root / "missing_sensor_fold_results.csv"
    fold_df.to_csv(fold_csv, index=False)

    summary_df = _ms_summarize(fold_df)
    summary_csv = save_root / "missing_sensor_summary.csv"
    summary_df.to_csv(summary_csv, index=False)

    _ms_plot(summary_df, save_root)

    print("\nRebuilt from saved runs:")
    print("Saved:", fold_csv)
    print("Saved:", summary_csv)
    print("Saved plots in:", save_root)

    return {
        "fold_df": fold_df,
        "summary_df": summary_df,
    }

def run_missing_sensor_ablation_resume(
    base_cfg: Config,
    missing_rates: List[float],
    seeds: List[int],
    save_root: str,
    skip_completed: bool = True,
):
    """
    Resume-safe runner.
    If a run already has:
        save_root / missing_XX / seed_YY / results.json
    it will be skipped.

    After finishing, it rebuilds:
      - missing_sensor_fold_results.csv
      - missing_sensor_summary.csv
      - plots
    from all saved runs.
    """
    save_root = Path(save_root)
    save_root.mkdir(parents=True, exist_ok=True)

    for missing_rate in missing_rates:
        variant = MissingSensorVariant(
            name=f"missing_{int(round(100 * missing_rate)):02d}",
            description=f"{int(round(100 * missing_rate))}% held-out channels missing",
            missing_sensor_rate=float(missing_rate),
        )

        for seed in seeds:
            run_dir = save_root / variant.name / f"seed_{seed}"
            results_json = run_dir / "results.json"

            if skip_completed and results_json.exists():
                print(f"[SKIP] already completed -> rate={missing_rate:.2f}, seed={seed}")
                continue

            cfg = copy.deepcopy(base_cfg)
            cfg.seed = int(seed)

            print(f"\n######## RESUME MISSING SENSOR | rate={missing_rate:.2f} | seed={seed} ########")
            out = run_loso_experiment_missing_sensor_variant(cfg, variant)

            run_dir.mkdir(parents=True, exist_ok=True)
            _ms_safe_save_json(out, results_json)
            print(f"[SAVED] {results_json}")

    return rebuild_missing_sensor_outputs(str(save_root))

# missing_sensor_results = run_missing_sensor_ablation_resume(
#     base_cfg=base_cfg,
#     missing_rates=[0.30],
#     seeds=[42],
#     save_root=str(Path(base_cfg.save_dir) / "missing_sensor_ablation"),
#     skip_completed=True,
# )
#
# support_results = run_support_sweep(
#     base_cfg=base_cfg,
#     support_values=[0.05, 0.10, 0.20, 0.30],
#     seeds=[42],
#     save_root=str(Path(base_cfg.save_dir) / "support_sweep"),
# )
#
# missing_sensor_results = run_missing_sensor_ablation(
#     base_cfg=base_cfg,
#     missing_rates=[0.0, 0.10, 0.20, 0.30],
#     seeds=[42],
#     save_root=str(Path(base_cfg.save_dir) / "missing_sensor_ablation"),
# )
#
# support_results = run_support_sweep(
#     base_cfg=base_cfg,
#     support_values=[0.05, 0.10, 0.20, 0.30],
#     seeds=[42],
#     save_root=str(Path(base_cfg.save_dir) / "support_sweep"),
# )
#
#
# from google.colab import drive
# drive.flush_and_unmount()
#
# from google.colab import runtime
# runtime.unassign()
